# GrantScopeAI — CORDIS Cleaning and Integration

## Notebook Purpose

This notebook prepares the Horizon Europe CORDIS data for use in GrantScopeAI.

The raw CORDIS package contains one central project table and several supporting tables describing participating organisations, scientific classifications, funding topics, programme information, and project links. These tables must be cleaned, aggregated, and integrated before they can support exploratory analysis, topic filtering, and the similar-grants recommendation system.

The primary objective is to create a reliable project-level dataset containing one row per funded Horizon Europe project.

## Project Context

GrantScopeAI is an exploratory research-intelligence tool designed to help researchers and university research-support professionals understand how an early AI-enabled chemistry or materials proposal fits within recent funding and publication activity.

The project combines:

- CORDIS for European Union-funded research projects
- NSF Award Search for United States research awards
- OpenAlex for publication activity and research-momentum context

This notebook focuses only on the CORDIS component.

## Source Files

The Horizon Europe bulk-data package contains six related CSV files:

- `project.csv` — central project information
- `organization.csv` — participating organisations, countries, roles, and contributions
- `euroSciVoc.csv` — scientific classifications
- `topics.csv` — official Horizon funding topics
- `legalBasis.csv` — programme and legal-framework information
- `webLink.csv` — CORDIS and external project links

The central relationship is:

> `project.csv["id"]` → supporting tables’ `projectID`

A standardized field named `project_id_clean` will be used for all joins because the raw identifiers are stored in different formats across the source tables.

## Main Cleaning Objectives

This notebook will:

1. Load all six raw CORDIS CSV files.
2. Apply the controlled parsing process required for `project.csv`.
3. Standardize column names and project identifiers.
4. Validate that the central project table contains one row per project.
5. Parse project dates and funding amounts.
6. Standardize missing values and text fields.
7. Aggregate the one-to-many supporting tables to one row per project.
8. Merge the aggregated supporting information into the central project table.
9. Confirm that the merge does not multiply project rows.
10. Restrict the analytical dataset to projects starting from 2021 through 2025.
11. Create preliminary AI and chemistry/materials relevance indicators.
12. Produce project-level data-quality checks.
13. Save an interim CORDIS candidate dataset for later analysis and modelling.

## Planned Project-Level Fields

The enriched CORDIS dataset is expected to include:

### Project information

- Project identifier
- Acronym
- Title
- Objective
- Keywords
- Project status
- Start and end dates
- Record year
- Framework programme
- Funding scheme
- Calls and funding topics

### Funding information

- Total project cost
- Maximum European Commission contribution
- Native currency: EUR

### Organisation and geography information

- Coordinator organisation
- Coordinator country
- Participating organisation count
- Participating country count
- Participating countries
- Organisation names
- Organisation-level contribution summaries

### Scientific classification

- EuroSciVoc titles
- EuroSciVoc paths
- Classification count
- Horizon topic codes
- Horizon topic titles

### Traceability

- Original CORDIS project link
- Source name
- Source record identifier
- Extraction or source-update information

## Integration Strategy

The supporting tables contain multiple rows for many projects. They will therefore be aggregated before merging.

The intended sequence is:

 Projects → EuroSciVoc summary → Topic summary → Organisation summary → Legal-basis summary → Preferred web link

All merges will be left joins from the central project table.

The project row count and uniqueness of `project_id_clean` will be checked after every merge. Any increase in the number of project rows will be treated as a possible many-to-many join error.

## Date Scope

The primary GrantScopeAI analytical period is 2021–2025.

The complete Horizon Europe project table will be retained during cleaning, while a separate scoped dataset will be created for projects whose start dates fall within this period.

Projects outside the selected period will not be deleted from the raw source files.

## Initial Relevance Scope

The preliminary relevance filter will identify projects involving both:

### Artificial-intelligence or computational methods

Examples include:

- Artificial intelligence
- Machine learning
- Deep learning
- Neural networks
- Scientific machine learning
- Data-driven methods
- Computer vision
- Natural-language processing
- Autonomous laboratories
- Self-driving laboratories

### Chemistry, materials, or molecular science

Examples include:

- Chemistry
- Chemical sciences
- Molecular modelling
- Molecules
- Materials science
- Materials discovery
- Catalysis
- Reactions
- Polymers
- Spectroscopy
- Computational chemistry
- Laboratory automation

Evidence may come from project titles, objectives, keywords, EuroSciVoc classifications, or Horizon topic descriptions.

The first relevance filter will create a candidate dataset rather than make a final scientific classification. Included and excluded examples will be reviewed manually before the filter is finalized.

## Data-Quality Principles

The cleaning process will follow these rules:

- Preserve the raw files unchanged.
- Retain original identifiers and source fields.
- Create standardized fields rather than overwriting source values unnecessarily.
- Document malformed rows and parser assumptions.
- Do not silently discard projects.
- Do not merge raw one-to-many tables directly.
- Record row counts before and after each major transformation.
- Preserve missing supporting information as null values.
- Keep EUR funding amounts separate from NSF funding reported in USD.
- Avoid exposing unnecessary personal contact information.
- Distinguish source-derived values from engineered fields.

## Expected Outputs

This notebook will produce:

- A cleaned central Horizon Europe project table
- Aggregated project-level supporting tables
- An enriched one-row-per-project CORDIS dataset
- A 2021–2025 project-scope dataset
- Preliminary relevance flags
- A CORDIS data-quality summary
- An interim candidate CSV saved in `Data/Processed_Data`

The final CORDIS output will later be standardized with the NSF grant dataset for exploratory analysis and the TF-IDF similar-project recommendation system.

## 1. Load Raw CORDIS Tables

The Horizon Europe package contains one central project table and five
supporting tables.

Each CSV is loaded into a separate DataFrame. The supporting tables will
remain separate until they have been cleaned and aggregated to one row
per project.

All source columns are initially loaded as strings to preserve identifiers
and avoid unintended type conversion. Dates and numeric funding fields
will be converted during the cleaning stage.

In [1]:
from pathlib import Path
import csv

import pandas as pd


PROJECT_ROOT = Path(
    r"C:\Users\kahau\OneDrive\Documents\Techmeup"
    r"\Final_Project\GrantScopeAI"
)

HORIZON_EUROPE_DIR = (
    PROJECT_ROOT
    / "Data"
    / "Raw_Data"
    / "CORDIS"
    / "Horizon_Europe"
)

print("CORDIS directory:")
print(HORIZON_EUROPE_DIR.resolve())
print("Directory exists:", HORIZON_EUROPE_DIR.exists())

CORDIS directory:
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\GrantScopeAI\Data\Raw_Data\CORDIS\Horizon_Europe
Directory exists: True


In [2]:
CORDIS_FILES = {
    "projects": HORIZON_EUROPE_DIR / "project.csv",
    "organizations": HORIZON_EUROPE_DIR / "organization.csv",
    "euroSciVoc": HORIZON_EUROPE_DIR / "euroSciVoc.csv",
    "topics": HORIZON_EUROPE_DIR / "topics.csv",
    "legalBasis": HORIZON_EUROPE_DIR / "legalBasis.csv",
    "webLinks": HORIZON_EUROPE_DIR / "webLink.csv"
}

for table_name, file_path in CORDIS_FILES.items():
    print(
        f"{table_name}: "
        f"{file_path.exists()} — "
        f"{file_path.name}"
    )

projects: True — project.csv
organizations: True — organization.csv
euroSciVoc: True — euroSciVoc.csv
topics: True — topics.csv
legalBasis: True — legalBasis.csv
webLinks: True — webLink.csv


In [3]:
def load_standard_cordis_csv(file_path):
    """
    Load a standard semicolon-delimited CORDIS source table.

    All columns are initially retained as strings so that source
    identifiers and formatting are preserved during acquisition.
    """
    return pd.read_csv(
        file_path,
        sep=";",
        encoding="utf-8-sig",
        dtype="string",
        low_memory=False
    )

In [4]:
EXPECTED_PROJECT_COLUMNS = 22
OBJECTIVE_INDEX = 15
TRAILING_COLUMNS_AFTER_OBJECTIVE = 6

unrepaired_project_rows = []


def repair_cordis_project_row(fields):
    """
    Repair a malformed CORDIS project row when extra semicolons
    have split the objective field into multiple pieces.
    """
    if len(fields) == EXPECTED_PROJECT_COLUMNS:
        return fields

    if len(fields) > EXPECTED_PROJECT_COLUMNS:
        objective_end = (
            len(fields)
            - TRAILING_COLUMNS_AFTER_OBJECTIVE
        )

        repaired_fields = (
            fields[:OBJECTIVE_INDEX]
            + [
                ";".join(
                    fields[OBJECTIVE_INDEX:objective_end]
                )
            ]
            + fields[objective_end:]
        )

        if len(repaired_fields) == EXPECTED_PROJECT_COLUMNS:
            return repaired_fields

    unrepaired_project_rows.append(fields)
    return None

In [5]:
cordis_projects_df = pd.read_csv(
    CORDIS_FILES["projects"],
    sep=";",
    encoding="utf-8-sig",
    engine="python",
    quoting=csv.QUOTE_NONE,
    on_bad_lines=repair_cordis_project_row,
    dtype="string"
)

print(
    f"Projects loaded: "
    f"{cordis_projects_df.shape[0]:,} rows × "
    f"{cordis_projects_df.shape[1]} columns"
)

print(
    "Rows that could not be repaired:",
    len(unrepaired_project_rows)
)

Projects loaded: 23,278 rows × 22 columns
Rows that could not be repaired: 0


In [6]:
cordis_projects_df

,"""id""","""acronym""","""status""","""title""","""startDate""","""endDate""","""totalCost""","""ecMaxContribution""","""topics""","""ecSignatureDate""",...,"""subCall""","""fundingScheme""","""nature""","""objective""","""contentUpdateDate""","""rcn""","""grantDoi""","""keywords""","""Human-validated""","""legalBasis"""
0,"""101069359""","""SolDAC""","""SIGNED""","""Full spectrum SOLar Direct Air Capture & conv...","""2022-09-01""","""2025-08-31""","""2073781,25""","""2073781,25""","""HORIZON-CL5-2021-D2-01-11""","""2022-05-12""",...,"""HORIZON-CL5-2021-D2-01""","""HORIZON-RIA""","""""","""Ethylene is the chemical industry’s primary b...","""2026-03-27 15:10:23""","""237915""","""10.3030/101069359""","""zero-carbon, solar energy, air mining, Negati...","""true""","""HORIZON.2.5"""
1,"""101069357""","""Photo2Fuel""","""SIGNED""","""Artificial PHOTOsynthesis to produce FUELs an...","""2022-09-01""","""2025-08-31""","""2493171,25""","""2493171""","""HORIZON-CL5-2021-D2-01-08""","""2022-05-18""",...,"""HORIZON-CL5-2021-D2-01""","""HORIZON-RIA""","""""","""The Photo2Fuel project will develop a breakth...",products separation,acetic acid,methane,"energy storage""","""true""","""HORIZON.2.5"""
2,"""101069586""","""BOLSTER""","""SIGNED""","""Bridging Organizations and marginalized commu...","""2022-09-01""","""2025-08-31""","""3792955""","""3792955""","""HORIZON-CL5-2021-D2-01-12""","""2022-06-10""",...,"""HORIZON-CL5-2021-D2-01""","""HORIZON-RIA""","""""","""No European should be put at a disadvantaged ...","""2025-09-30 15:22:06""","""237917""","""10.3030/101069586""","""Transition strategies, Participation, Margi...","""true""","""HORIZON.2.5"""
3,"""101069604""","""GREENET""","""SIGNED""","""NCPS NETwork for the GREEN transition in clim...","""2022-07-01""","""2027-12-31""","""3250901,25""","""3250901,25""","""HORIZON-CL5-2021-D2-01-15""","""2022-06-09""",...,"""HORIZON-CL5-2021-D2-01""","""HORIZON-CSA""","""""","""The overall aim of GREENET is to improve the ...","""2026-06-22 12:35:17""","""237918""","""10.3030/101069604""","""Competence development, Capacity building, CL...","""NA""","""HORIZON.2.5"""
4,"""101069529""","""SSH CENTRE""","""SIGNED""","""Social Sciences and Humanities for Climate, E...","""2022-09-01""","""2026-02-28""","""2305696,25""","""2305696""","""HORIZON-CL5-2021-D2-01-13""","""2022-05-20""",...,"""HORIZON-CL5-2021-D2-01""","""HORIZON-CSA""","""""","""The Social Sciences & Humanities for Climate,...",Research and innovation,EU Green Deal,Horizon Europe,"EU SET-Plan""","""true""","""HORIZON.2.5"""
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23273,"""101334839""","""LOFlu_TREAT""","""SIGNED""","""Targeting liquid organelles to develop new an...","""2027-01-01""","""2028-06-30""","""0""","""150000""","""ERC-2026-POC""","""2026-07-13""",...,"""ERC-2026-POC""","""HORIZON-ERC-POC""","""""","""Rising antiviral resistance, vaccine hesitanc...","""2026-07-20 15:54:32""","""292186""","""10.3030/101334839""","""""","""false""","""HORIZON.1.1"""
23274,"""101334858""","""TBX4-PRECIS""","""SIGNED""","""TBX4 enhancer based precision gene targeting ...","""2026-08-01""","""2028-01-31""","""0""","""150000""","""ERC-2026-POC""","""2026-07-14""",...,"""ERC-2026-POC""","""HORIZON-ERC-POC""","""""","""Chronic respiratory diseases (CRDs) represent...","""2026-07-20 15:54:32""","""292187""","""10.3030/101334858""","""""","""false""","""HORIZON.1.1"""
23275,"""101334879""","""NAIVE""","""SIGNED""","""Neotropical AI-assisted VErtebrate identifica...","""2026-12-01""","""2028-05-31""","""0""","""150000""","""ERC-2026-POC""","""2026-07-14""",...,"""ERC-2026-POC""","""HORIZON-ERC-POC""","""""","""The biodiversity crisis in the Neotropics, dr...","""2026-07-20 15:54:35""","""292188""","""10.3030/101334879""","""""","""false""","""HORIZON.1.1"""
23276,"""101334897""","""DETEQ""","""SIGNED""","""Ultrafast Superconducting Nanowire Single-Pho...","""2026-10-01""","""2028-03-31""","""0""","""150

In [7]:
def remove_matching_outer_quotes(value):
    """
    Remove matching quotation marks around a string while
    preserving quotation marks inside the value.
    """
    if pd.isna(value):
        return value

    value = str(value).strip()

    if (
        len(value) >= 2
        and value.startswith('"')
        and value.endswith('"')
    ):
        value = value[1:-1]

    return value.replace('""', '"')

In [8]:
for column in cordis_projects_df.columns:
    cordis_projects_df[column] = (
        cordis_projects_df[column]
        .map(remove_matching_outer_quotes)
        .astype("string")
    )

In [18]:
cordis_projects_df.columns = (
    cordis_projects_df.columns
    .str.strip()
    .str.strip('"')
)

print("Cleaned project columns:")
print(cordis_projects_df.columns.tolist())

print(
    "First five project IDs:",
    cordis_projects_df["id"].head().tolist()
)

Cleaned project columns:
['id', 'acronym', 'status', 'title', 'startDate', 'endDate', 'totalCost', 'ecMaxContribution', 'topics', 'ecSignatureDate', 'frameworkProgramme', 'masterCall', 'subCall', 'fundingScheme', 'nature', 'objective', 'contentUpdateDate', 'rcn', 'grantDoi', 'keywords', 'Human-validated', 'legalBasis']
First five project IDs: ['101069359', '101069357', '101069586', '101069604', '101069529']


In [9]:
cordis_projects_df

,"""id""","""acronym""","""status""","""title""","""startDate""","""endDate""","""totalCost""","""ecMaxContribution""","""topics""","""ecSignatureDate""",...,"""subCall""","""fundingScheme""","""nature""","""objective""","""contentUpdateDate""","""rcn""","""grantDoi""","""keywords""","""Human-validated""","""legalBasis"""
0,101069359,SolDAC,SIGNED,Full spectrum SOLar Direct Air Capture & conve...,2022-09-01,2025-08-31,"2073781,25","2073781,25",HORIZON-CL5-2021-D2-01-11,2022-05-12,...,HORIZON-CL5-2021-D2-01,HORIZON-RIA,,Ethylene is the chemical industry’s primary bu...,2026-03-27 15:10:23,237915,10.3030/101069359,"zero-carbon, solar energy, air mining, Negativ...",true,HORIZON.2.5
1,101069357,Photo2Fuel,SIGNED,Artificial PHOTOsynthesis to produce FUELs and...,2022-09-01,2025-08-31,"2493171,25",2493171,HORIZON-CL5-2021-D2-01-08,2022-05-18,...,HORIZON-CL5-2021-D2-01,HORIZON-RIA,,"""The Photo2Fuel project will develop a breakth...",products separation,acetic acid,methane,"energy storage""",true,HORIZON.2.5
2,101069586,BOLSTER,SIGNED,Bridging Organizations and marginalized commun...,2022-09-01,2025-08-31,3792955,3792955,HORIZON-CL5-2021-D2-01-12,2022-06-10,...,HORIZON-CL5-2021-D2-01,HORIZON-RIA,,No European should be put at a disadvantaged b...,2025-09-30 15:22:06,237917,10.3030/101069586,"Transition strategies, Participation, Margin...",true,HORIZON.2.5
3,101069604,GREENET,SIGNED,NCPS NETwork for the GREEN transition in clima...,2022-07-01,2027-12-31,"3250901,25","3250901,25",HORIZON-CL5-2021-D2-01-15,2022-06-09,...,HORIZON-CL5-2021-D2-01,HORIZON-CSA,,The overall aim of GREENET is to improve the p...,2026-06-22 12:35:17,237918,10.3030/101069604,"Competence development, Capacity building, CL5...",NA,HORIZON.2.5
4,101069529,SSH CENTRE,SIGNED,"Social Sciences and Humanities for Climate, En...",2022-09-01,2026-02-28,"2305696,25",2305696,HORIZON-CL5-2021-D2-01-13,2022-05-20,...,HORIZON-CL5-2021-D2-01,HORIZON-CSA,,"""The Social Sciences & Humanities for Climate,...",Research and innovation,EU Green Deal,Horizon Europe,"EU SET-Plan""",true,HORIZON.2.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23273,101334839,LOFlu_TREAT,SIGNED,Targeting liquid organelles to develop new ant...,2027-01-01,2028-06-30,0,150000,ERC-2026-POC,2026-07-13,...,ERC-2026-POC,HORIZON-ERC-POC,,"Rising antiviral resistance, vaccine hesitancy...",2026-07-20 15:54:32,292186,10.3030/101334839,,false,HORIZON.1.1
23274,101334858,TBX4-PRECIS,SIGNED,TBX4 enhancer based precision gene targeting f...,2026-08-01,2028-01-31,0,150000,ERC-2026-POC,2026-07-14,...,ERC-2026-POC,HORIZON-ERC-POC,,Chronic respiratory diseases (CRDs) represent ...,2026-07-20 15:54:32,292187,10.3030/101334858,,false,HORIZON.1.1
23275,101334879,NAIVE,SIGNED,Neotropical AI-assisted VErtebrate identificat...,2026-12-01,2028-05-31,0,150000,ERC-2026-POC,2026-07-14,...,ERC-2026-POC,HORIZON-ERC-POC,,"The biodiversity crisis in the Neotropics, dri...",2026-07-20 15:54:35,292188,10.3030/101334879,,false,HORIZON.1.1
23276,101334897,DETEQ,SIGNED,Ultrafast Superconducting Nanowire Single-Phot...,2026-10-01,2028-03-31,0,150000,ERC-2026-POC,2026-07-14,...,ERC-2026-POC,HORIZON-ERC-POC,,"""In photonic quantum technologies, quantum inf...",2026-07-20 15:54:34,292189,10.3030/101334897,,false,HORIZON.1.1


In [19]:
cordis_organizations_df = pd.read_csv(
    CORDIS_FILES["organizations"],
    sep=";",
    encoding="utf-8-sig",
    dtype="string",
    low_memory=False
)

print(
    f"Organizations loaded: "
    f"{cordis_organizations_df.shape[0]:,} rows × "
    f"{cordis_organizations_df.shape[1]} columns"
)

Organizations loaded: 144,117 rows × 25 columns


In [20]:
cordis_scivoc_df = pd.read_csv(
    CORDIS_FILES["euroSciVoc"],
    sep=";",
    encoding="utf-8-sig",
    dtype="string",
    low_memory=False
)

print(
    f"EuroSciVoc loaded: "
    f"{cordis_scivoc_df.shape[0]:,} rows × "
    f"{cordis_scivoc_df.shape[1]} columns"
)

EuroSciVoc loaded: 51,718 rows × 5 columns


In [21]:
cordis_topics_df = pd.read_csv(
    CORDIS_FILES["topics"],
    sep=";",
    encoding="utf-8-sig",
    dtype="string",
    low_memory=False
)

print(
    f"Topics loaded: "
    f"{cordis_topics_df.shape[0]:,} rows × "
    f"{cordis_topics_df.shape[1]} columns"
)

Topics loaded: 23,278 rows × 3 columns


In [22]:
cordis_legal_df = pd.read_csv(
    CORDIS_FILES["legalBasis"],
    sep=";",
    encoding="utf-8-sig",
    dtype="string",
    low_memory=False
)

print(
    f"Legal basis loaded: "
    f"{cordis_legal_df.shape[0]:,} rows × "
    f"{cordis_legal_df.shape[1]} columns"
)

Legal basis loaded: 30,044 rows × 4 columns


In [23]:
cordis_links_df = pd.read_csv(
    CORDIS_FILES["webLinks"],
    sep=";",
    encoding="utf-8-sig",
    dtype="string",
    low_memory=False
)

print(
    f"Web links loaded: "
    f"{cordis_links_df.shape[0]:,} rows × "
    f"{cordis_links_df.shape[1]} columns"
)

Web links loaded: 57,859 rows × 9 columns


In [24]:
print("Project table shape:", cordis_projects_df.shape)
print("Project columns:")
print(cordis_projects_df.columns.tolist())

Project table shape: (23278, 22)
Project columns:
['id', 'acronym', 'status', 'title', 'startDate', 'endDate', 'totalCost', 'ecMaxContribution', 'topics', 'ecSignatureDate', 'frameworkProgramme', 'masterCall', 'subCall', 'fundingScheme', 'nature', 'objective', 'contentUpdateDate', 'rcn', 'grantDoi', 'keywords', 'Human-validated', 'legalBasis']


In [25]:
print(
    f"Projects loaded: "
    f"{cordis_projects_df.shape[0]:,} rows × "
    f"{cordis_projects_df.shape[1]} columns"
)

print(
    "First five project IDs:",
    cordis_projects_df["id"].head().tolist()
)

Projects loaded: 23,278 rows × 22 columns
First five project IDs: ['101069359', '101069357', '101069586', '101069604', '101069529']


In [26]:
cordis_dataframes = {
    "projects": cordis_projects_df,
    "organizations": cordis_organizations_df,
    "euroSciVoc": cordis_scivoc_df,
    "topics": cordis_topics_df,
    "legalBasis": cordis_legal_df,
    "webLinks": cordis_links_df
}

for table_name, dataframe in cordis_dataframes.items():
    print(
        f"{table_name}: "
        f"{dataframe.shape[0]:,} rows × "
        f"{dataframe.shape[1]} columns"
    )

projects: 23,278 rows × 22 columns
organizations: 144,117 rows × 25 columns
euroSciVoc: 51,718 rows × 5 columns
topics: 23,278 rows × 3 columns
legalBasis: 30,044 rows × 4 columns
webLinks: 57,859 rows × 9 columns


In [27]:
id_columns = {
    "projects": cordis_projects_df["id"],
    "organizations": cordis_organizations_df["projectID"],
    "euroSciVoc": cordis_scivoc_df["projectID"],
    "topics": cordis_topics_df["projectID"],
    "legalBasis": cordis_legal_df["projectID"],
    "webLinks": cordis_links_df["projectID"]
}

for table_name, id_series in id_columns.items():
    print(f"\n{table_name}")
    print("Data type:", id_series.dtype)
    print("Examples:", id_series.dropna().head(3).tolist())


projects
Data type: string
Examples: ['101069359', '101069357', '101069586']

organizations
Data type: string
Examples: ['101069359', '101069359', '101069359']

euroSciVoc
Data type: string
Examples: ['101069359', '101069359', '101069359']

topics
Data type: string
Examples: ['101069359', '101069357', '101069586']

legalBasis
Data type: string
Examples: ['101069359', '101069357', '101069357']

webLinks
Data type: string
Examples: ['101069359', '101069359', '101069359']


In [28]:
def clean_cordis_project_id(series):
    """
    Standardize CORDIS project identifiers for joins.
    """
    return (
        series
        .astype("string")
        .str.strip()
        .str.strip('"')
        .str.strip("'")
        .str.replace(r"\.0$", "", regex=True)
    )


cordis_projects_df["project_id_clean"] = clean_cordis_project_id(
    cordis_projects_df["id"]
)

cordis_organizations_df["project_id_clean"] = clean_cordis_project_id(
    cordis_organizations_df["projectID"]
)

cordis_scivoc_df["project_id_clean"] = clean_cordis_project_id(
    cordis_scivoc_df["projectID"]
)

cordis_topics_df["project_id_clean"] = clean_cordis_project_id(
    cordis_topics_df["projectID"]
)

cordis_legal_df["project_id_clean"] = clean_cordis_project_id(
    cordis_legal_df["projectID"]
)

cordis_links_df["project_id_clean"] = clean_cordis_project_id(
    cordis_links_df["projectID"]
)

print("Created project_id_clean in all six CORDIS tables.")

Created project_id_clean in all six CORDIS tables.


In [29]:
join_key_summary = pd.DataFrame([
    {
        "table": "projects",
        "rows": len(cordis_projects_df),
        "missing_ids": cordis_projects_df["project_id_clean"].isna().sum(),
        "unique_projects": cordis_projects_df["project_id_clean"].nunique(),
        "duplicate_rows": cordis_projects_df["project_id_clean"].duplicated().sum()
    },
    {
        "table": "organizations",
        "rows": len(cordis_organizations_df),
        "missing_ids": cordis_organizations_df["project_id_clean"].isna().sum(),
        "unique_projects": cordis_organizations_df["project_id_clean"].nunique(),
        "duplicate_rows": cordis_organizations_df["project_id_clean"].duplicated().sum()
    },
    {
        "table": "euroSciVoc",
        "rows": len(cordis_scivoc_df),
        "missing_ids": cordis_scivoc_df["project_id_clean"].isna().sum(),
        "unique_projects": cordis_scivoc_df["project_id_clean"].nunique(),
        "duplicate_rows": cordis_scivoc_df["project_id_clean"].duplicated().sum()
    },
    {
        "table": "topics",
        "rows": len(cordis_topics_df),
        "missing_ids": cordis_topics_df["project_id_clean"].isna().sum(),
        "unique_projects": cordis_topics_df["project_id_clean"].nunique(),
        "duplicate_rows": cordis_topics_df["project_id_clean"].duplicated().sum()
    },
    {
        "table": "legalBasis",
        "rows": len(cordis_legal_df),
        "missing_ids": cordis_legal_df["project_id_clean"].isna().sum(),
        "unique_projects": cordis_legal_df["project_id_clean"].nunique(),
        "duplicate_rows": cordis_legal_df["project_id_clean"].duplicated().sum()
    },
    {
        "table": "webLinks",
        "rows": len(cordis_links_df),
        "missing_ids": cordis_links_df["project_id_clean"].isna().sum(),
        "unique_projects": cordis_links_df["project_id_clean"].nunique(),
        "duplicate_rows": cordis_links_df["project_id_clean"].duplicated().sum()
    }
])

display(join_key_summary)

,table,rows,missing_ids,unique_projects,duplicate_rows
0,projects,23278,0,23278,0
1,organizations,144117,0,23278,120839
2,euroSciVoc,51718,0,20062,31656
3,topics,23278,0,23278,0
4,legalBasis,30044,0,23278,6766
5,webLinks,57859,0,11081,46778


In [30]:
project_fields_to_inspect = [
    "startDate",
    "endDate",
    "ecSignatureDate",
    "contentUpdateDate",
    "totalCost",
    "ecMaxContribution"
]

for column in project_fields_to_inspect:
    print(f"\n{column}")
    print("Data type:", cordis_projects_df[column].dtype)
    print(
        "Examples:",
        cordis_projects_df[column]
        .dropna()
        .head(5)
        .tolist()
    )


startDate
Data type: string
Examples: ['2022-09-01', '2022-09-01', '2022-09-01', '2022-07-01', '2022-09-01']

endDate
Data type: string
Examples: ['2025-08-31', '2025-08-31', '2025-08-31', '2027-12-31', '2026-02-28']

ecSignatureDate
Data type: string
Examples: ['2022-05-12', '2022-05-18', '2022-06-10', '2022-06-09', '2022-05-20']

contentUpdateDate
Data type: string
Examples: ['2026-03-27 15:10:23', 'products separation', '2025-09-30 15:22:06', '2026-06-22 12:35:17', 'Research and innovation']

totalCost
Data type: string
Examples: ['2073781,25', '2493171,25', '3792955', '3250901,25', '2305696,25']

ecMaxContribution
Data type: string
Examples: ['2073781,25', '2493171', '3792955', '3250901,25', '2305696']


In [32]:
first_invalid_index = cordis_projects_df.index[
    invalid_update_date_mask
][0]

print("First invalid row index:", first_invalid_index)

display(
    cordis_projects_df.loc[
        first_invalid_index
    ].to_frame(name="value")
)

First invalid row index: 1


,value
id,101069357
acronym,Photo2Fuel
status,SIGNED
title,Artificial PHOTOsynthesis to produce FUELs and...
startDate,2022-09-01
endDate,2025-08-31
totalCost,"2493171,25"
ecMaxContribution,2493171
topics,HORIZON-CL5-2021-D2-01-08
ecSignatureDate,2022-05-18


In [33]:
project_id_to_inspect = "101069357"

with open(
    CORDIS_FILES["projects"],
    mode="r",
    encoding="utf-8-sig"
) as file:
    matching_lines = [
        line
        for line in file
        if project_id_to_inspect in line
    ]

print("Matching raw lines:", len(matching_lines))

if matching_lines:
    raw_line = matching_lines[0]

    print("\nRaw line:")
    print(raw_line)

    print("\nNumber of semicolons:")
    print(raw_line.count(";"))

    print("\nLine length:")
    print(len(raw_line))

Matching raw lines: 1

Raw line:
"101069357";"Photo2Fuel";"SIGNED";"Artificial PHOTOsynthesis to produce FUELs and chemicals: hybrid systems with microorganisms for improved light harvesting and CO2 reduction";"2022-09-01";"2025-08-31";"2493171,25";"2493171";"HORIZON-CL5-2021-D2-01-08";"2022-05-18";"HORIZON";"HORIZON-CL5-2021-D2-01";"HORIZON-CL5-2021-D2-01";"HORIZON-RIA";"";"The Photo2Fuel project will develop a breakthrough technology that converts CO2 into useful fuels and chemicals by means of non-photosynthetic microorganisms and organic materials, using only sunlight as energy source. Photo2Fuel's technology is based on the artificial photosynthesis concept and will use a hybrid system of non-photosynthetic microorganisms and organic photosensitisers to produce acetic acid and methane, using Moorella thermoacetica (bacteria) and Methanosarcina barkeri (archaea) strains, respectively. After optimisation and characterisation, this hybrid non-photosynthetic microorganisms with organi

In [34]:
project_path = CORDIS_FILES["projects"]

expected_column_count = 22
parsed_rows = []
problem_rows = []

with open(
    project_path,
    mode="r",
    encoding="utf-8-sig"
) as file:

    # Parse the header
    header_line = file.readline().rstrip("\r\n")
    header = header_line[1:-1].split('";"')

    # Parse each project row
    for line_number, line in enumerate(file, start=2):
        line = line.rstrip("\r\n")

        if line.startswith('"') and line.endswith('"'):
            fields = line[1:-1].split('";"')
        else:
            fields = []

        if len(fields) == expected_column_count:
            fields = [
                value.replace('""', '"')
                for value in fields
            ]
            parsed_rows.append(fields)
        else:
            problem_rows.append({
                "line_number": line_number,
                "field_count": len(fields),
                "raw_line": line
            })

cordis_projects_df = pd.DataFrame(
    parsed_rows,
    columns=header,
    dtype="string"
)

print(
    f"Projects loaded: "
    f"{cordis_projects_df.shape[0]:,} rows × "
    f"{cordis_projects_df.shape[1]} columns"
)

print("Problem rows:", len(problem_rows))
print(
    "Invalid contentUpdateDate values:",
    (
        ~cordis_projects_df["contentUpdateDate"].str.match(
            r"^\d{4}-\d{2}-\d{2}( \d{2}:\d{2}:\d{2})?$",
            na=False
        )
    ).sum()
)

Projects loaded: 23,278 rows × 22 columns
Problem rows: 0
Invalid contentUpdateDate values: 0


In [35]:
cordis_projects_df["project_id_clean"] = (
    cordis_projects_df["id"]
    .astype("string")
    .str.strip()
    .str.strip('"')
    .str.strip("'")
    .str.replace(r"\.0$", "", regex=True)
)

print(
    "Projects:",
    f"{len(cordis_projects_df):,}"
)

print(
    "Unique clean project IDs:",
    f"{cordis_projects_df['project_id_clean'].nunique():,}"
)

print(
    "Duplicate clean IDs:",
    cordis_projects_df["project_id_clean"].duplicated().sum()
)

Projects: 23,278
Unique clean project IDs: 23,278
Duplicate clean IDs: 0


In [36]:
cordis_date_columns = [
    "startDate",
    "endDate",
    "ecSignatureDate",
    "contentUpdateDate"
]

for column in cordis_date_columns:
    cordis_projects_df[column] = pd.to_datetime(
        cordis_projects_df[column],
        errors="coerce"
    )

for column in cordis_date_columns:
    print(
        f"{column}: "
        f"{cordis_projects_df[column].isna().sum():,} missing/invalid | "
        f"dtype = {cordis_projects_df[column].dtype}"
    )

startDate: 0 missing/invalid | dtype = datetime64[ns]
endDate: 0 missing/invalid | dtype = datetime64[ns]
ecSignatureDate: 1 missing/invalid | dtype = datetime64[ns]
contentUpdateDate: 0 missing/invalid | dtype = datetime64[ns]


In [37]:
display(
    cordis_projects_df.loc[
        cordis_projects_df["ecSignatureDate"].isna(),
        [
            "project_id_clean",
            "acronym",
            "title",
            "status",
            "startDate",
            "ecSignatureDate"
        ]
    ]
)

,project_id_clean,acronym,title,status,startDate,ecSignatureDate
20545,101294070,ScaleUA,ScaleUA: A Roadmap for Rebuilding and Integrat...,SIGNED,2026-06-01,NaT


In [38]:
funding_columns = [
    "totalCost",
    "ecMaxContribution"
]

for column in funding_columns:
    cordis_projects_df[column] = pd.to_numeric(
        cordis_projects_df[column]
        .str.replace(",", ".", regex=False)
        .str.strip(),
        errors="coerce"
    )

for column in funding_columns:
    print(
        f"{column}: "
        f"{cordis_projects_df[column].isna().sum():,} missing/invalid | "
        f"dtype = {cordis_projects_df[column].dtype}"
    )

totalCost: 0 missing/invalid | dtype = Float64
ecMaxContribution: 0 missing/invalid | dtype = Float64


In [39]:
cordis_projects_df["record_year"] = (
    cordis_projects_df["startDate"].dt.year
)

print(
    "Earliest project year:",
    cordis_projects_df["record_year"].min()
)

print(
    "Latest project year:",
    cordis_projects_df["record_year"].max()
)

display(
    cordis_projects_df["record_year"]
    .value_counts()
    .sort_index()
    .rename_axis("record_year")
    .reset_index(name="project_count")
)

Earliest project year: 2021
Latest project year: 2027


,record_year,project_count
0,2021,30
1,2022,3344
2,2023,5130
3,2024,4959
4,2025,4468
5,2026,4546
6,2027,801


In [40]:
cordis_projects_df["in_project_date_scope"] = (
    cordis_projects_df["record_year"]
    .between(2021, 2025)
)

cordis_scope_df = (
    cordis_projects_df[
        cordis_projects_df["in_project_date_scope"]
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "All Horizon Europe projects:",
    f"{len(cordis_projects_df):,}"
)

print(
    "Projects in 2021–2025 scope:",
    f"{len(cordis_scope_df):,}"
)

print(
    "Projects outside scope:",
    f"{(~cordis_projects_df['in_project_date_scope']).sum():,}"
)

All Horizon Europe projects: 23,278
Projects in 2021–2025 scope: 17,931
Projects outside scope: 5,347


In [41]:
core_project_fields = [
    "project_id_clean",
    "title",
    "objective",
    "keywords",
    "topics",
    "startDate",
    "endDate",
    "totalCost",
    "ecMaxContribution",
    "fundingScheme"
]

project_missingness = pd.DataFrame({
    "column": core_project_fields,
    "missing_count": [
        cordis_projects_df[column].isna().sum()
        for column in core_project_fields
    ],
    "missing_percent": [
        round(
            cordis_projects_df[column].isna().mean() * 100,
            2
        )
        for column in core_project_fields
    ]
})

display(project_missingness)

,column,missing_count,missing_percent
0,project_id_clean,0,0.0
1,title,0,0.0
2,objective,0,0.0
3,keywords,0,0.0
4,topics,0,0.0
5,startDate,0,0.0
6,endDate,0,0.0
7,totalCost,0,0.0
8,ecMaxContribution,0,0.0
9,fundingScheme,0,0.0


In [42]:
cordis_projects_df["search_text"] = (
    cordis_projects_df["title"].fillna("")
    + " "
    + cordis_projects_df["objective"].fillna("")
    + " "
    + cordis_projects_df["keywords"].fillna("")
    + " "
    + cordis_projects_df["topics"].fillna("")
)

cordis_projects_df["search_text"] = (
    cordis_projects_df["search_text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print(
    "Search text created for:",
    f"{cordis_projects_df['search_text'].notna().sum():,} projects"
)

print(
    "Empty search-text rows:",
    cordis_projects_df["search_text"].eq("").sum()
)

print(
    "Example search-text length:",
    len(cordis_projects_df.loc[0, "search_text"])
)

Search text created for: 23,278 projects
Empty search-text rows: 0
Example search-text length: 2248


In [43]:
organization_role_counts = (
    cordis_organizations_df["role"]
    .value_counts(dropna=False)
    .rename_axis("role")
    .reset_index(name="row_count")
)

display(organization_role_counts)

,role,row_count
0,participant,90239
1,associatedPartner,24423
2,coordinator,23278
3,thirdParty,6177


In [44]:
cordis_coordinators_df = (
    cordis_organizations_df[
        cordis_organizations_df["role"].eq("coordinator")
    ][
        [
            "project_id_clean",
            "organisationID",
            "name",
            "shortName",
            "activityType",
            "city",
            "country",
            "SME",
            "ecContribution",
            "netEcContribution",
            "totalCost"
        ]
    ]
    .copy()
    .rename(
        columns={
            "organisationID": "coordinator_organisation_id",
            "name": "coordinator_name",
            "shortName": "coordinator_short_name",
            "activityType": "coordinator_activity_type",
            "city": "coordinator_city",
            "country": "coordinator_country",
            "SME": "coordinator_is_sme",
            "ecContribution": "coordinator_ec_contribution",
            "netEcContribution": "coordinator_net_ec_contribution",
            "totalCost": "coordinator_total_cost"
        }
    )
    .reset_index(drop=True)
)

print(
    "Coordinator rows:",
    f"{len(cordis_coordinators_df):,}"
)

print(
    "Unique coordinator project IDs:",
    f"{cordis_coordinators_df['project_id_clean'].nunique():,}"
)

print(
    "Duplicate coordinator project IDs:",
    cordis_coordinators_df["project_id_clean"].duplicated().sum()
)

Coordinator rows: 23,278
Unique coordinator project IDs: 23,278
Duplicate coordinator project IDs: 0


In [45]:
organization_type_counts = (
    cordis_organizations_df["activityType"]
    .value_counts(dropna=False)
    .rename_axis("activity_type")
    .reset_index(name="row_count")
)

display(organization_type_counts)

,activity_type,row_count
0,HES,51653
1,PRC,41800
2,REC,31571
3,OTH,11555
4,PUB,7537
5,<NA>,1


In [48]:
missing_by_role = (
    cordis_organizations_df
    .groupby("role", dropna=False)
    .agg(
        rows=("project_id_clean", "size"),
        missing_ec_contribution=(
            "ecContribution",
            lambda values: values.isna().sum()
        ),
        missing_net_ec_contribution=(
            "netEcContribution",
            lambda values: values.isna().sum()
        ),
        missing_total_cost=(
            "totalCost",
            lambda values: values.isna().sum()
        ),
        missing_sme=(
            "SME",
            lambda values: values.isna().sum()
        )
    )
    .reset_index()
)

display(missing_by_role)

,role,rows,missing_ec_contribution,missing_net_ec_contribution,missing_total_cost,missing_sme
0,associatedPartner,24423,24423.0,11.0,44.0,127
1,coordinator,23278,0.0,0.0,0.0,0
2,participant,90239,0.0,0.0,0.0,234
3,thirdParty,6177,0.0,0.0,0.0,6


In [49]:
missing_sme_by_activity = (
    cordis_organizations_df[
        cordis_organizations_df["SME"].isna()
    ]
    .groupby(
        ["activityType", "role"],
        dropna=False
    )
    .size()
    .reset_index(name="missing_sme_rows")
    .sort_values(
        "missing_sme_rows",
        ascending=False
    )
)

display(missing_sme_by_activity)

,activityType,role,missing_sme_rows
3,PRC,participant,234
2,PRC,associatedPartner,116
4,PRC,thirdParty,6
1,OTH,associatedPartner,4
0,HES,associatedPartner,4
6,REC,associatedPartner,2
5,PUB,associatedPartner,1


In [50]:
duplicate_project_org_mask = (
    cordis_organizations_df
    .duplicated(
        subset=[
            "project_id_clean",
            "organisationID"
        ],
        keep=False
    )
)

print(
    "Duplicate project–organisation rows:",
    f"{duplicate_project_org_mask.sum():,}"
)

print(
    "Affected projects:",
    cordis_organizations_df.loc[
        duplicate_project_org_mask,
        "project_id_clean"
    ].nunique()
)

display(
    cordis_organizations_df.loc[
        duplicate_project_org_mask,
        [
            "project_id_clean",
            "organisationID",
            "name",
            "role",
            "country"
        ]
    ].head(20)
)

Duplicate project–organisation rows: 498
Affected projects: 193


,project_id_clean,organisationID,name,role,country
294,101069890,889429570,ERION ENERGY,participant,IT
295,101069890,889429570,ERION ENERGY,thirdParty,IT
461,101069506,906446474,UNITED KINGDOM RESEARCH AND INNOVATION,associatedPartner,UK
462,101069506,906446474,UNITED KINGDOM RESEARCH AND INNOVATION,participant,UK
765,101040474,992204077,PRESIDENT AND FELLOWS OF HARVARD COLLEGE,participant,US
766,101040474,992204077,PRESIDENT AND FELLOWS OF HARVARD COLLEGE,thirdParty,US
1174,101046651,999465982,UNIVERSITE DE ROUEN NORMANDIE,thirdParty,FR
1175,101046651,999465982,UNIVERSITE DE ROUEN NORMANDIE,thirdParty,FR
1336,101047214,999848550,UNIVERSITE D'ORLEANS,coordinator,FR
1337,101047214,999848550,UNIVERSITE D'ORLEANS,thirdParty,FR


In [51]:
duplicate_project_org_summary = (
    cordis_organizations_df.loc[duplicate_project_org_mask]
    .groupby(
        ["project_id_clean", "organisationID"],
        dropna=False
    )
    .agg(
        row_count=("project_id_clean", "size"),
        roles=("role", lambda values: " | ".join(
            sorted(values.dropna().unique())
        )),
        names=("name", lambda values: " | ".join(
            sorted(values.dropna().unique())
        )),
        countries=("country", lambda values: " | ".join(
            sorted(values.dropna().unique())
        )),
        active_values=("active", lambda values: " | ".join(
            sorted(values.dropna().unique())
        ))
    )
    .reset_index()
    .sort_values(
        ["row_count", "project_id_clean"],
        ascending=[False, True]
    )
)

print(
    "Repeated project–organisation combinations:",
    f"{len(duplicate_project_org_summary):,}"
)

display(
    duplicate_project_org_summary.head(20)
)

Repeated project–organisation combinations: 249


,project_id_clean,organisationID,row_count,roles,names,countries,active_values
0,101039402,999854758,2,participant | thirdParty,ECOLE NORMALE SUPERIEURE,FR,
1,101040474,992204077,2,participant | thirdParty,PRESIDENT AND FELLOWS OF HARVARD COLLEGE,US,
2,101046203,999544455,2,participant | thirdParty,FUNDACIO CENTRE DE REGULACIO GENOMICA,ES,
3,101046489,909875521,2,associatedPartner | thirdParty,SORBONNE UNIVERSITE,FR,
4,101046651,999465982,2,thirdParty,UNIVERSITE DE ROUEN NORMANDIE,FR,
5,101047214,999848550,2,coordinator | thirdParty,UNIVERSITE D'ORLEANS,FR,
6,101054369,999990267,2,associatedPartner | participant,MAX-PLANCK-GESELLSCHAFT ZUR FORDERUNG DER WISS...,DE,
7,101054957,999981343,2,associatedPartner | participant,GENOME RESEARCH LIMITED LBG,UK,
8,101055286,890009048,2,associatedPartner | participant,SCIENTIFIC AND INNOVATION PARTNERSHIP ASSISTAN...,AM,
9,101055286,906324254,2,associatedPartner | participant,AGENTIA NATIONALA PENTRU CERCETARE SI DEZVOLTARE,MD,


In [52]:
duplicate_rows_df = cordis_organizations_df.loc[
    duplicate_project_org_mask
].copy()

duplicate_pattern_df = (
    duplicate_rows_df
    .groupby(
        ["project_id_clean", "organisationID"],
        dropna=False
    )
    .agg(
        row_count=("project_id_clean", "size"),
        distinct_roles=("role", "nunique"),
        distinct_names=("name", "nunique"),
        distinct_countries=("country", "nunique"),
        distinct_active_values=("active", "nunique")
    )
    .reset_index()
)

exact_duplicate_mask = duplicate_rows_df.duplicated(
    keep=False
)

print(
    "Repeated combinations:",
    f"{len(duplicate_pattern_df):,}"
)

print(
    "Combinations with one role:",
    f"{duplicate_pattern_df['distinct_roles'].eq(1).sum():,}"
)

print(
    "Combinations with multiple roles:",
    f"{duplicate_pattern_df['distinct_roles'].gt(1).sum():,}"
)

print(
    "Completely identical duplicate rows:",
    f"{exact_duplicate_mask.sum():,}"
)

print("\nRows per repeated combination:")
display(
    duplicate_pattern_df["row_count"]
    .value_counts()
    .sort_index()
    .rename_axis("rows_per_combination")
    .reset_index(name="combination_count")
)

Repeated combinations: 249
Combinations with one role: 44
Combinations with multiple roles: 205
Completely identical duplicate rows: 0

Rows per repeated combination:


,rows_per_combination,combination_count
0,2,249


In [53]:
single_role_keys_df = duplicate_pattern_df.loc[
    duplicate_pattern_df["distinct_roles"].eq(1),
    ["project_id_clean", "organisationID"]
]

single_role_duplicates_df = duplicate_rows_df.merge(
    single_role_keys_df,
    on=["project_id_clean", "organisationID"],
    how="inner"
)

comparison_columns = [
    column
    for column in cordis_organizations_df.columns
    if column not in [
        "project_id_clean",
        "organisationID"
    ]
]

single_role_difference_counts = (
    single_role_duplicates_df
    .groupby(
        ["project_id_clean", "organisationID"],
        dropna=False
    )[comparison_columns]
    .nunique(dropna=False)
    .gt(1)
    .sum()
    .sort_values(ascending=False)
    .rename("combinations_with_different_values")
    .reset_index()
)

display(
    single_role_difference_counts[
        single_role_difference_counts[
            "combinations_with_different_values"
        ].gt(0)
    ]
)

,index,combinations_with_different_values
0,order,44
1,netEcContribution,41
2,endOfParticipation,32
3,ecContribution,13
4,totalCost,10


In [54]:
same_role_duplicate_sample = (
    single_role_duplicates_df[
        [
            "project_id_clean",
            "organisationID",
            "name",
            "role",
            "order",
            "ecContribution",
            "netEcContribution",
            "totalCost",
            "endOfParticipation",
            "active"
        ]
    ]
    .sort_values(
        [
            "project_id_clean",
            "organisationID",
            "order"
        ]
    )
    .head(20)
)

display(same_role_duplicate_sample)

,project_id_clean,organisationID,name,role,order,ecContribution,netEcContribution,totalCost,endOfParticipation,active
0,101046651,999465982,UNIVERSITE DE ROUEN NORMANDIE,thirdParty,1,0.0,251080.0,0.0,false,<NA>
1,101046651,999465982,UNIVERSITE DE ROUEN NORMANDIE,thirdParty,5,0.0,47252.5,0.0,false,<NA>
2,101058522,889809519,ERION WEEE,thirdParty,1,0.0,0.0,0.0,true,<NA>
3,101058522,889809519,ERION WEEE,thirdParty,27,0.0,20000.0,0.0,false,<NA>
8,101058548,891452990,SITEDRIVE OY,thirdParty,18,0.0,0.0,0.0,true,<NA>
9,101058548,891452990,SITEDRIVE OY,thirdParty,22,0.0,65621.25,0.0,false,<NA>
12,101058572,880988630,UNIVERSITATEA NATIONALA DE STIINTASI TEHNOLOGI...,participant,7,48753.45,48753.45,48753.45,true,<NA>
13,101058572,880988630,UNIVERSITATEA NATIONALA DE STIINTASI TEHNOLOGI...,participant,9,90621.55,90621.55,90621.55,false,<NA>
7,101060212,894742357,SEENOVIA,thirdParty,55,0.0,0.0,0.0,true,<NA>
6,101060212,894742357,SEENOVIA,thirdParty,68,0.0,79499.5,0.0,false,<NA>


In [56]:
def join_values_for_display(values):
    return " | ".join(
        "<NA>" if pd.isna(value) else str(value)
        for value in values
    )


multi_role_keys_df = duplicate_pattern_df.loc[
    duplicate_pattern_df["distinct_roles"].gt(1),
    ["project_id_clean", "organisationID"]
]

multi_role_duplicates_df = duplicate_rows_df.merge(
    multi_role_keys_df,
    on=["project_id_clean", "organisationID"],
    how="inner"
)

multi_role_summary_df = (
    multi_role_duplicates_df
    .groupby(
        ["project_id_clean", "organisationID"],
        dropna=False
    )
    .agg(
        organisation_name=("name", "first"),
        roles=(
            "role",
            lambda values: " | ".join(
                sorted(values.dropna().astype(str).unique())
            )
        ),
        ec_contributions=(
            "ecContribution",
            join_values_for_display
        ),
        net_ec_contributions=(
            "netEcContribution",
            join_values_for_display
        ),
        total_costs=(
            "totalCost",
            join_values_for_display
        )
    )
    .reset_index()
)

print(
    "Multiple-role combinations:",
    f"{len(multi_role_summary_df):,}"
)

display(
    multi_role_summary_df["roles"]
    .value_counts()
    .rename_axis("role_combination")
    .reset_index(name="combination_count")
)

display(multi_role_summary_df.head(10))

Multiple-role combinations: 205


,role_combination,combination_count
0,associatedPartner | participant,133
1,participant | thirdParty,59
2,associatedPartner | thirdParty,4
3,coordinator | thirdParty,4
4,associatedPartner | coordinator,3
5,coordinator | participant,2


,project_id_clean,organisationID,organisation_name,roles,ec_contributions,net_ec_contributions,total_costs
0,101039402,999854758,ECOLE NORMALE SUPERIEURE,participant | thirdParty,112791.25 | 0.0,112791.25 | 364873.95,112791.25 | 0.0
1,101040474,992204077,PRESIDENT AND FELLOWS OF HARVARD COLLEGE,participant | thirdParty,0.0 | 0.0,0.0 | 131250.0,0.0 | 0.0
2,101046203,999544455,FUNDACIO CENTRE DE REGULACIO GENOMICA,participant | thirdParty,260000.0 | 0.0,260000.0 | 0.0,260000.0 | 0.0
3,101046489,909875521,SORBONNE UNIVERSITE,associatedPartner | thirdParty,<NA> | 0.0,0.0 | 125220.0,0.0 | 0.0
4,101047214,999848550,UNIVERSITE D'ORLEANS,coordinator | thirdParty,373189.25 | 0.0,373189.25 | 48327.94,373189.25 | 0.0
5,101054369,999990267,MAX-PLANCK-GESELLSCHAFT ZUR FORDERUNG DER WISS...,associatedPartner | participant,<NA> | 205000.0,0.0 | 205000.0,0.0 | 205000.0
6,101054957,999981343,GENOME RESEARCH LIMITED LBG,associatedPartner | participant,<NA> | 393387.5,0.0 | 393387.5,0.0 | 393387.5
7,101055286,890009048,SCIENTIFIC AND INNOVATION PARTNERSHIP ASSISTAN...,associatedPartner | participant,<NA> | 197681.25,0.0 | 197681.25,0.0 | 197681.25
8,101055286,906324254,AGENTIA NATIONALA PENTRU CERCETARE SI DEZVOLTARE,associatedPartner | participant,<NA> | 165625.0,0.0 | 165625.0,0.0 | 165625.0
9,101055286,911084335,AGJENCIA KOMBETARE E KERKIMIT SHKENCOR DHE INO...,associatedPartner | participant,<NA> | 165656.25,0.0 | 165656.25,0.0 | 165656.25


In [57]:
organization_funding_check_df = (
    cordis_organizations_df
    .groupby("project_id_clean", dropna=False)
    .agg(
        organisation_row_count=(
            "organisationID",
            "size"
        ),
        unique_organisation_count=(
            "organisationID",
            "nunique"
        ),
        summed_ec_contribution=(
            "ecContribution",
            lambda values: values.sum(min_count=1)
        ),
        summed_net_ec_contribution=(
            "netEcContribution",
            lambda values: values.sum(min_count=1)
        ),
        summed_organisation_total_cost=(
            "totalCost",
            lambda values: values.sum(min_count=1)
        )
    )
    .reset_index()
    .merge(
        cordis_projects_df[
            [
                "project_id_clean",
                "ecMaxContribution",
                "totalCost"
            ]
        ],
        on="project_id_clean",
        how="left",
        validate="one_to_one"
    )
)

organization_funding_check_df["ec_difference"] = (
    organization_funding_check_df["summed_ec_contribution"]
    - organization_funding_check_df["ecMaxContribution"]
).abs()

organization_funding_check_df["net_ec_difference"] = (
    organization_funding_check_df["summed_net_ec_contribution"]
    - organization_funding_check_df["ecMaxContribution"]
).abs()

organization_funding_check_df["total_cost_difference"] = (
    organization_funding_check_df["summed_organisation_total_cost"]
    - organization_funding_check_df["totalCost"]
).abs()

print("Projects checked:", f"{len(organization_funding_check_df):,}")

print(
    "Summed ecContribution within €1 of project value:",
    f"{organization_funding_check_df['ec_difference'].le(1).sum():,}"
)

print(
    "Summed netEcContribution within €1 of project value:",
    f"{organization_funding_check_df['net_ec_difference'].le(1).sum():,}"
)

print(
    "Summed organisation totalCost within €1 of project value:",
    f"{organization_funding_check_df['total_cost_difference'].le(1).sum():,}"
)

display(
    organization_funding_check_df[
        [
            "project_id_clean",
            "ecMaxContribution",
            "summed_ec_contribution",
            "summed_net_ec_contribution",
            "ec_difference",
            "net_ec_difference"
        ]
    ]
    .sort_values("net_ec_difference", ascending=False)
    .head(10)
)

Projects checked: 23,278
Summed ecContribution within €1 of project value: 23,231
Summed netEcContribution within €1 of project value: 23,228
Summed organisation totalCost within €1 of project value: 23,234


,project_id_clean,ecMaxContribution,summed_ec_contribution,summed_net_ec_contribution,ec_difference,net_ec_difference
866,101052200,549442000.0,664587862.11,548197398.76,115145862.11,1244601.24
259,101041177,1998008.75,1421349.82,1421349.82,576658.93,576658.93
4745,101083409,3924195.0,3373043.75,3373043.75,551151.25,551151.25
19681,101264088,420751.08,420751.08,736760.52,0.0,316009.44
21169,101282068,432467.52,432467.52,748476.96,0.0,316009.44
17122,101210078,434048.88,434048.88,742007.76,0.0,307958.88
20910,101279399,430343.4,430343.4,730512.84,0.0,300169.44
20996,101280264,396991.08,396991.08,697160.52,0.0,300169.44
17102,101209987,396991.08,396991.08,697160.52,0.0,300169.44
20424,101274267,397206.72,397206.72,697376.16,0.0,300169.44


In [58]:
identity_consistency_df = (
    duplicate_rows_df
    .groupby(
        ["project_id_clean", "organisationID"],
        dropna=False
    )
    .agg(
        distinct_names=("name", lambda x: x.nunique(dropna=False)),
        distinct_countries=("country", lambda x: x.nunique(dropna=False)),
        distinct_cities=("city", lambda x: x.nunique(dropna=False)),
        distinct_activity_types=(
            "activityType",
            lambda x: x.nunique(dropna=False)
        ),
        distinct_sme_values=(
            "SME",
            lambda x: x.nunique(dropna=False)
        )
    )
    .reset_index()
)

identity_difference_summary = pd.DataFrame({
    "field": [
        "name",
        "country",
        "city",
        "activityType",
        "SME"
    ],
    "repeated_combinations_with_conflicts": [
        identity_consistency_df["distinct_names"].gt(1).sum(),
        identity_consistency_df["distinct_countries"].gt(1).sum(),
        identity_consistency_df["distinct_cities"].gt(1).sum(),
        identity_consistency_df["distinct_activity_types"].gt(1).sum(),
        identity_consistency_df["distinct_sme_values"].gt(1).sum()
    ]
})

display(identity_difference_summary)

,field,repeated_combinations_with_conflicts
0,name,0
1,country,0
2,city,0
3,activityType,0
4,SME,1


In [59]:
sme_conflict_keys_df = identity_consistency_df.loc[
    identity_consistency_df["distinct_sme_values"].gt(1),
    ["project_id_clean", "organisationID"]
]

sme_conflict_rows_df = (
    duplicate_rows_df
    .merge(
        sme_conflict_keys_df,
        on=["project_id_clean", "organisationID"],
        how="inner"
    )
    [
        [
            "project_id_clean",
            "organisationID",
            "name",
            "role",
            "activityType",
            "SME",
            "order",
            "ecContribution",
            "netEcContribution",
            "totalCost"
        ]
    ]
    .sort_values(
        ["project_id_clean", "organisationID", "order"]
    )
)

display(sme_conflict_rows_df)

,project_id_clean,organisationID,name,role,activityType,SME,order,ecContribution,netEcContribution,totalCost
1,101130915,903762872,INKODE SOCIETA COOPERATIVA,thirdParty,PRC,False,34,0.0,0.0,0.0
0,101130915,903762872,INKODE SOCIETA COOPERATIVA,participant,PRC,True,42,91500.0,91500.0,91500.0


In [60]:
organisation_identity_check = pd.DataFrame({
    "field": [
        "organisationID",
        "name",
        "country",
        "activityType"
    ],
    "missing_rows": [
        cordis_organizations_df["organisationID"].isna().sum(),
        cordis_organizations_df["name"].isna().sum(),
        cordis_organizations_df["country"].isna().sum(),
        cordis_organizations_df["activityType"].isna().sum()
    ]
})

display(organisation_identity_check)

print(
    "Projects containing a missing organisationID:",
    cordis_organizations_df.loc[
        cordis_organizations_df["organisationID"].isna(),
        "project_id_clean"
    ].nunique()
)

,field,missing_rows
0,organisationID,0
1,name,0
2,country,18
3,activityType,1


Projects containing a missing organisationID: 0


In [61]:
def consolidate_sme_status(values):
    non_missing_values = values.dropna()

    if non_missing_values.eq(True).any():
        return True

    if non_missing_values.eq(False).any():
        return False

    return pd.NA


cordis_unique_organisations_df = (
    cordis_organizations_df
    .groupby(
        ["project_id_clean", "organisationID"],
        dropna=False
    )
    .agg(
        organisation_name=("name", "first"),
        organisation_short_name=("shortName", "first"),
        activity_type=("activityType", "first"),
        city=("city", "first"),
        country=("country", "first"),
        is_sme=("SME", consolidate_sme_status),
        sme_status_conflict=(
            "SME",
            lambda values: values.nunique(dropna=True) > 1
        ),
        roles=(
            "role",
            lambda values: " | ".join(
                sorted(values.dropna().astype(str).unique())
            )
        ),
        organisation_row_count=("organisationID", "size"),
        ec_contribution=(
            "ecContribution",
            lambda values: values.sum(min_count=1)
        ),
        net_ec_contribution=(
            "netEcContribution",
            lambda values: values.sum(min_count=1)
        ),
        organisation_total_cost=(
            "totalCost",
            lambda values: values.sum(min_count=1)
        )
    )
    .reset_index()
)

cordis_unique_organisations_df["is_sme"] = (
    cordis_unique_organisations_df["is_sme"]
    .astype("boolean")
)

print(
    "Original organisation rows:",
    f"{len(cordis_organizations_df):,}"
)

print(
    "Unique project–organisation rows:",
    f"{len(cordis_unique_organisations_df):,}"
)

print(
    "Duplicate project–organisation keys remaining:",
    cordis_unique_organisations_df.duplicated(
        subset=["project_id_clean", "organisationID"]
    ).sum()
)

print(
    "SME conflicts retained:",
    f"{cordis_unique_organisations_df['sme_status_conflict'].sum():,}"
)

Original organisation rows: 144,117
Unique project–organisation rows: 143,868
Duplicate project–organisation keys remaining: 0
SME conflicts retained: 1


In [62]:
# Create organisation-category and role indicators
cordis_unique_organisations_df["is_higher_education"] = (
    cordis_unique_organisations_df["activity_type"].eq("HES")
)

cordis_unique_organisations_df["is_private_company"] = (
    cordis_unique_organisations_df["activity_type"].eq("PRC")
)

cordis_unique_organisations_df["is_research_organisation"] = (
    cordis_unique_organisations_df["activity_type"].eq("REC")
)

cordis_unique_organisations_df["is_public_body"] = (
    cordis_unique_organisations_df["activity_type"].eq("PUB")
)

cordis_unique_organisations_df["has_coordinator_role"] = (
    cordis_unique_organisations_df["roles"]
    .str.contains("coordinator", na=False)
)

cordis_unique_organisations_df["has_participant_role"] = (
    cordis_unique_organisations_df["roles"]
    .str.contains("participant", na=False)
)

cordis_unique_organisations_df["has_associated_partner_role"] = (
    cordis_unique_organisations_df["roles"]
    .str.contains("associatedPartner", na=False)
)

cordis_unique_organisations_df["has_third_party_role"] = (
    cordis_unique_organisations_df["roles"]
    .str.contains("thirdParty", na=False)
)


cordis_organisation_summary_df = (
    cordis_unique_organisations_df
    .groupby("project_id_clean", dropna=False)
    .agg(
        organisation_count=("organisationID", "size"),
        country_count=("country", "nunique"),
        countries=(
            "country",
            lambda values: " | ".join(
                sorted(values.dropna().astype(str).unique())
            )
        ),
        organisation_names=(
            "organisation_name",
            lambda values: " | ".join(
                sorted(values.dropna().astype(str).unique())
            )
        ),
        higher_education_count=("is_higher_education", "sum"),
        private_company_count=("is_private_company", "sum"),
        research_organisation_count=(
            "is_research_organisation",
            "sum"
        ),
        public_body_count=("is_public_body", "sum"),
        sme_count=(
            "is_sme",
            lambda values: values.eq(True).sum()
        ),
        unknown_sme_count=(
            "is_sme",
            lambda values: values.isna().sum()
        ),
        coordinator_organisation_count=(
            "has_coordinator_role",
            "sum"
        ),
        participant_organisation_count=(
            "has_participant_role",
            "sum"
        ),
        associated_partner_organisation_count=(
            "has_associated_partner_role",
            "sum"
        ),
        third_party_organisation_count=(
            "has_third_party_role",
            "sum"
        ),
        sme_status_conflict_count=(
            "sme_status_conflict",
            "sum"
        )
    )
    .reset_index()
)

print(
    "Organisation summary rows:",
    f"{len(cordis_organisation_summary_df):,}"
)

print(
    "Unique project IDs:",
    f"{cordis_organisation_summary_df['project_id_clean'].nunique():,}"
)

print(
    "Duplicate project IDs:",
    cordis_organisation_summary_df[
        "project_id_clean"
    ].duplicated().sum()
)

display(cordis_organisation_summary_df.head())

Organisation summary rows: 23,278
Unique project IDs: 23,278
Duplicate project IDs: 0


,project_id_clean,organisation_count,country_count,countries,organisation_names,higher_education_count,private_company_count,research_organisation_count,public_body_count,sme_count,unknown_sme_count,coordinator_organisation_count,participant_organisation_count,associated_partner_organisation_count,third_party_organisation_count,sme_status_conflict_count
0,101039048,1,1,ES,FUNDACION DONOSTIA INTERNATIONAL PHYSICS CENTER,0,0,1,0,0,0,1,0,0,0,0
1,101039060,3,1,ES,ARANZADI ZIENTZI ELKARTEA | UNIVERSIDAD DEL PA...,2,0,1,0,0,0,1,2,0,0,0
2,101039066,1,1,PL,UNIWERSYTET IM. ADAMA MICKIEWICZA WPOZNANIU,1,0,0,0,0,0,1,0,0,0,0
3,101039090,1,1,DE,MAX-PLANCK-GESELLSCHAFT ZUR FORDERUNG DER WISS...,0,0,1,0,0,0,1,0,0,0,0
4,101039098,1,1,DE,UNIVERSITY OF HAMBURG,1,0,0,0,0,0,1,0,0,0,0


In [64]:
coordinator_dtype_check = (
    cordis_coordinators_df[
        [
            "coordinator_is_sme",
            "coordinator_ec_contribution",
            "coordinator_net_ec_contribution",
            "coordinator_total_cost"
        ]
    ]
    .dtypes
    .rename("dtype")
    .reset_index()
    .rename(columns={"index": "column"})
)

display(coordinator_dtype_check)

,column,dtype
0,coordinator_is_sme,boolean
1,coordinator_ec_contribution,Float64
2,coordinator_net_ec_contribution,Float64
3,coordinator_total_cost,Float64


In [65]:
cordis_projects_enriched_df = (
    cordis_projects_df
    .merge(
        cordis_coordinators_df,
        on="project_id_clean",
        how="left",
        validate="one_to_one"
    )
    .merge(
        cordis_organisation_summary_df,
        on="project_id_clean",
        how="left",
        validate="one_to_one"
    )
)

print(
    "Original project rows:",
    f"{len(cordis_projects_df):,}"
)

print(
    "Enriched project rows:",
    f"{len(cordis_projects_enriched_df):,}"
)

print(
    "Duplicate project IDs:",
    cordis_projects_enriched_df[
        "project_id_clean"
    ].duplicated().sum()
)

print(
    "Projects missing coordinator:",
    cordis_projects_enriched_df[
        "coordinator_name"
    ].isna().sum()
)

print(
    "Projects missing organisation summary:",
    cordis_projects_enriched_df[
        "organisation_count"
    ].isna().sum()
)

print(
    "Total enriched columns:",
    cordis_projects_enriched_df.shape[1]
)

Original project rows: 23,278
Enriched project rows: 23,278
Duplicate project IDs: 0
Projects missing coordinator: 0
Projects missing organisation summary: 0
Total enriched columns: 51


In [66]:
def join_unique_nonempty(values):
    cleaned_values = (
        values
        .dropna()
        .astype(str)
        .str.strip()
    )

    cleaned_values = cleaned_values[
        cleaned_values.ne("")
    ].unique()

    return " | ".join(sorted(cleaned_values))


cordis_scivoc_summary_df = (
    cordis_scivoc_df
    .groupby(
        "project_id_clean",
        dropna=False
    )
    .agg(
        scivoc_term_count=(
            "euroSciVocCode",
            "nunique"
        ),
        scivoc_titles=(
            "euroSciVocTitle",
            join_unique_nonempty
        ),
        scivoc_paths=(
            "euroSciVocPath",
            join_unique_nonempty
        )
    )
    .reset_index()
)

print(
    "EuroSciVoc summary rows:",
    f"{len(cordis_scivoc_summary_df):,}"
)

print(
    "Unique project IDs:",
    f"{cordis_scivoc_summary_df['project_id_clean'].nunique():,}"
)

print(
    "Duplicate project IDs:",
    cordis_scivoc_summary_df[
        "project_id_clean"
    ].duplicated().sum()
)

print(
    "Projects with EuroSciVoc classifications:",
    f"{cordis_scivoc_summary_df['project_id_clean'].nunique():,}"
)

display(cordis_scivoc_summary_df.head())

EuroSciVoc summary rows: 20,062
Unique project IDs: 20,062
Duplicate project IDs: 0
Projects with EuroSciVoc classifications: 20,062


,project_id_clean,scivoc_term_count,scivoc_titles,scivoc_paths
0,101039048,2,neutrinos | noble gases,natural sciences/chemical sciences/inorganic c...
1,101039060,5,bioarchaeology | computational science | nutri...,humanities/history and archaeology/archaeology...
2,101039066,3,climatic changes | ecosystems | forestry,"agricultural sciences/agriculture, forestry, a..."
3,101039090,3,didactics | reinforcement learning | virtual r...,natural sciences/computer and information scie...
4,101039098,1,quantum physics,natural sciences/physical sciences/quantum phy...


In [67]:
cordis_topic_details_df = (
    cordis_topics_df[
        [
            "project_id_clean",
            "topic",
            "title"
        ]
    ]
    .copy()
    .rename(
        columns={
            "topic": "call_topic_code",
            "title": "call_topic_title"
        }
    )
    .reset_index(drop=True)
)

print(
    "Topic-detail rows:",
    f"{len(cordis_topic_details_df):,}"
)

print(
    "Unique project IDs:",
    f"{cordis_topic_details_df['project_id_clean'].nunique():,}"
)

print(
    "Duplicate project IDs:",
    cordis_topic_details_df[
        "project_id_clean"
    ].duplicated().sum()
)

print(
    "Missing topic codes:",
    cordis_topic_details_df[
        "call_topic_code"
    ].isna().sum()
)

print(
    "Missing topic titles:",
    cordis_topic_details_df[
        "call_topic_title"
    ].isna().sum()
)

display(cordis_topic_details_df.head())

Topic-detail rows: 23,278
Unique project IDs: 23,278
Duplicate project IDs: 0
Missing topic codes: 0
Missing topic titles: 0


,project_id_clean,call_topic_code,call_topic_title
0,101069359,HORIZON-CL5-2021-D2-01-11,Direct atmospheric carbon capture and conversion
1,101069357,HORIZON-CL5-2021-D2-01-08,Emerging technologies for a climate neutral Eu...
2,101069586,HORIZON-CL5-2021-D2-01-12,Fostering a just transition in Europe
3,101069604,HORIZON-CL5-2021-D2-01-15,Fostering cooperation between Horizon Europe c...
4,101069529,HORIZON-CL5-2021-D2-01-13,Strengthening Social Sciences and Humanities (...


In [68]:
cordis_legal_summary_df = (
    cordis_legal_df
    .groupby(
        "project_id_clean",
        dropna=False
    )
    .agg(
        legal_basis_count=(
            "legalBasis",
            "nunique"
        ),
        legal_basis_codes=(
            "legalBasis",
            join_unique_nonempty
        ),
        legal_basis_titles=(
            "title",
            join_unique_nonempty
        ),
        programme_parts=(
            "uniqueProgrammePart",
            join_unique_nonempty
        )
    )
    .reset_index()
)

print(
    "Legal-basis summary rows:",
    f"{len(cordis_legal_summary_df):,}"
)

print(
    "Unique project IDs:",
    f"{cordis_legal_summary_df['project_id_clean'].nunique():,}"
)

print(
    "Duplicate project IDs:",
    cordis_legal_summary_df[
        "project_id_clean"
    ].duplicated().sum()
)

print(
    "Projects with multiple legal-basis codes:",
    f"{cordis_legal_summary_df['legal_basis_count'].gt(1).sum():,}"
)

display(cordis_legal_summary_df.head())

Legal-basis summary rows: 23,278
Unique project IDs: 23,278
Duplicate project IDs: 0
Projects with multiple legal-basis codes: 5,188


,project_id_clean,legal_basis_count,legal_basis_codes,legal_basis_titles,programme_parts
0,101039048,1,HORIZON.1.1,European Research Council (ERC),true
1,101039060,1,HORIZON.1.1,European Research Council (ERC),true
2,101039066,1,HORIZON.1.1,European Research Council (ERC),true
3,101039090,1,HORIZON.1.1,European Research Council (ERC),true
4,101039098,1,HORIZON.1.1,European Research Council (ERC),true


In [69]:
web_link_fields = [
    "type",
    "source",
    "represents",
    "status"
]

for column in web_link_fields:
    print(f"\n--- {column} ---")

    display(
        cordis_links_df[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="row_count")
        .head(20)
    )


--- type ---


,type,row_count
0,projectDeliverable,57054
1,socialMedia,523
2,relatedWebsite,270
3,relatedStory,9
4,relatedVideo,3



--- source ---


,source,row_count
0,corda,57054
1,editorial,794
2,rtd,11



--- represents ---


,represents,row_count
0,<NA>,51297
1,project,6038
2,linkedIn,191
3,twitter,154
4,youtube,75
5,facebook,60
6,instagram,39
7,mastodon,2
8,zenodo,2
9,researchGate,1



--- status ---


,status,row_count
0,<NA>,57824
1,legacy,32
2,webArchive,3


In [70]:
web_link_combination_counts = (
    cordis_links_df
    .groupby(
        ["type", "represents", "source"],
        dropna=False
    )
    .size()
    .reset_index(name="row_count")
    .sort_values(
        "row_count",
        ascending=False
    )
)

display(web_link_combination_counts.head(30))

,type,represents,source,row_count
1,projectDeliverable,<NA>,corda,51297
0,projectDeliverable,project,corda,5757
5,relatedWebsite,project,editorial,268
9,socialMedia,linkedIn,editorial,191
12,socialMedia,twitter,editorial,154
13,socialMedia,youtube,editorial,74
7,socialMedia,facebook,editorial,60
8,socialMedia,instagram,editorial,39
2,relatedStory,project,rtd,9
3,relatedVideo,project,editorial,2


In [71]:
project_website_candidates_df = (
    cordis_links_df.loc[
        cordis_links_df["type"].eq("relatedWebsite")
        & cordis_links_df["represents"].eq("project")
    ]
    .copy()
)

website_count_by_project_df = (
    project_website_candidates_df
    .groupby("project_id_clean", dropna=False)
    .agg(
        website_row_count=("physUrl", "size"),
        unique_website_count=("physUrl", "nunique"),
        website_urls=(
            "physUrl",
            join_unique_nonempty
        )
    )
    .reset_index()
)

print(
    "Project-website rows:",
    f"{len(project_website_candidates_df):,}"
)

print(
    "Projects with a website:",
    f"{website_count_by_project_df['project_id_clean'].nunique():,}"
)

print(
    "Projects with multiple unique websites:",
    f"{website_count_by_project_df['unique_website_count'].gt(1).sum():,}"
)

print(
    "Missing website URLs:",
    f"{project_website_candidates_df['physUrl'].isna().sum():,}"
)

display(
    website_count_by_project_df.loc[
        website_count_by_project_df["unique_website_count"].gt(1)
    ].head(20)
)

Project-website rows: 270
Projects with a website: 267
Projects with multiple unique websites: 3
Missing website URLs: 0


,project_id_clean,website_row_count,unique_website_count,website_urls
29,101059786,2,2,https://valuable-project.eu/ | https://web.arc...
55,101064984,2,2,https://web.archive.org/web/20240801195822/htt...
56,101066739,2,2,https://alejandoslomet.github.io/ | https://we...


In [72]:
multiple_website_project_ids = (
    website_count_by_project_df.loc[
        website_count_by_project_df["unique_website_count"].gt(1),
        "project_id_clean"
    ]
)

multiple_website_records_df = (
    project_website_candidates_df.loc[
        project_website_candidates_df["project_id_clean"].isin(
            multiple_website_project_ids
        ),
        [
            "project_id_clean",
            "physUrl",
            "source",
            "status",
            "archivedDate",
            "availableLanguages"
        ]
    ]
    .sort_values(
        ["project_id_clean", "physUrl"]
    )
)

display(multiple_website_records_df)

,project_id_clean,physUrl,source,status,archivedDate,availableLanguages
4169,101059786,https://valuable-project.eu/,editorial,legacy,2025-01-13 00:00:00,en
4170,101059786,https://web.archive.org/web/20241120070943/htt...,editorial,webArchive,2026-06-02 00:00:00,en
10382,101064984,https://web.archive.org/web/20240801195822/htt...,editorial,webArchive,2026-06-02 00:00:00,en
10381,101064984,https://www.evrisk.eu/,editorial,legacy,2025-08-04 00:00:00,en
10223,101066739,https://alejandoslomet.github.io/,editorial,legacy,2025-08-04 00:00:00,en
10218,101066739,https://web.archive.org/web/20240602182256/htt...,editorial,webArchive,2026-06-02 00:00:00,en


In [74]:
project_website_candidates_df = (
    cordis_links_df.loc[
        cordis_links_df["type"].eq("relatedWebsite")
        & cordis_links_df["represents"].eq("project")
    ]
    .copy()
)

# Missing status means "not identified as archived"
project_website_candidates_df["is_archive"] = (
    project_website_candidates_df["status"]
    .eq("webArchive")
    .fillna(False)
    |
    project_website_candidates_df["physUrl"]
    .str.contains(
        "web.archive.org",
        case=False,
        na=False
    )
)

website_counts_df = (
    project_website_candidates_df
    .groupby("project_id_clean")["physUrl"]
    .nunique()
    .rename("project_website_count")
    .reset_index()
)

live_websites_df = (
    project_website_candidates_df.loc[
        ~project_website_candidates_df["is_archive"],
        ["project_id_clean", "physUrl"]
    ]
    .drop_duplicates(
        subset="project_id_clean",
        keep="first"
    )
    .rename(
        columns={
            "physUrl": "project_website_url"
        }
    )
)

archive_websites_df = (
    project_website_candidates_df.loc[
        project_website_candidates_df["is_archive"],
        ["project_id_clean", "physUrl"]
    ]
    .drop_duplicates(
        subset="project_id_clean",
        keep="first"
    )
    .rename(
        columns={
            "physUrl": "project_website_archive_url"
        }
    )
)

cordis_website_summary_df = (
    website_counts_df
    .merge(
        live_websites_df,
        on="project_id_clean",
        how="left",
        validate="one_to_one"
    )
    .merge(
        archive_websites_df,
        on="project_id_clean",
        how="left",
        validate="one_to_one"
    )
)

print(
    "Website-summary rows:",
    f"{len(cordis_website_summary_df):,}"
)

print(
    "Duplicate project IDs:",
    cordis_website_summary_df[
        "project_id_clean"
    ].duplicated().sum()
)

print(
    "Projects with a live website:",
    f"{cordis_website_summary_df['project_website_url'].notna().sum():,}"
)

print(
    "Projects with an archived website:",
    f"{cordis_website_summary_df['project_website_archive_url'].notna().sum():,}"
)

display(
    cordis_website_summary_df.loc[
        cordis_website_summary_df["project_website_count"].gt(1)
    ]
)

Website-summary rows: 267
Duplicate project IDs: 0
Projects with a live website: 267
Projects with an archived website: 3


,project_id_clean,project_website_count,project_website_url,project_website_archive_url
29,101059786,2,https://valuable-project.eu/,https://web.archive.org/web/20241120070943/htt...
55,101064984,2,https://www.evrisk.eu/,https://web.archive.org/web/20240801195822/htt...
56,101066739,2,https://alejandoslomet.github.io/,https://web.archive.org/web/20240602182256/htt...


In [75]:
cordis_projects_full_df = (
    cordis_projects_enriched_df
    .merge(
        cordis_scivoc_summary_df,
        on="project_id_clean",
        how="left",
        validate="one_to_one"
    )
    .merge(
        cordis_topic_details_df,
        on="project_id_clean",
        how="left",
        validate="one_to_one"
    )
    .merge(
        cordis_legal_summary_df,
        on="project_id_clean",
        how="left",
        validate="one_to_one"
    )
    .merge(
        cordis_website_summary_df,
        on="project_id_clean",
        how="left",
        validate="one_to_one"
    )
)

print(
    "Rows before merges:",
    f"{len(cordis_projects_enriched_df):,}"
)

print(
    "Rows after merges:",
    f"{len(cordis_projects_full_df):,}"
)

print(
    "Duplicate project IDs:",
    cordis_projects_full_df[
        "project_id_clean"
    ].duplicated().sum()
)

print(
    "Projects missing EuroSciVoc:",
    f"{cordis_projects_full_df['scivoc_titles'].isna().sum():,}"
)

print(
    "Projects missing topic details:",
    f"{cordis_projects_full_df['call_topic_code'].isna().sum():,}"
)

print(
    "Projects missing legal-basis details:",
    f"{cordis_projects_full_df['legal_basis_codes'].isna().sum():,}"
)

print(
    "Projects missing a website:",
    f"{cordis_projects_full_df['project_website_url'].isna().sum():,}"
)

print(
    "Total columns:",
    cordis_projects_full_df.shape[1]
)

Rows before merges: 23,278
Rows after merges: 23,278
Duplicate project IDs: 0
Projects missing EuroSciVoc: 3,216
Projects missing topic details: 0
Projects missing legal-basis details: 0
Projects missing a website: 23,011
Total columns: 63


In [76]:
cordis_projects_full_df["cordis_project_url"] = (
    "https://cordis.europa.eu/project/id/"
    + cordis_projects_full_df["project_id_clean"].astype("string")
)

cordis_projects_full_df["has_external_project_website"] = (
    cordis_projects_full_df["project_website_url"].notna()
)

cordis_projects_full_df["preferred_project_url"] = (
    cordis_projects_full_df["project_website_url"]
    .fillna(cordis_projects_full_df["cordis_project_url"])
)

print(
    "Projects with an external website:",
    f"{cordis_projects_full_df['has_external_project_website'].sum():,}"
)

print(
    "Projects with an official CORDIS URL:",
    f"{cordis_projects_full_df['cordis_project_url'].notna().sum():,}"
)

print(
    "Projects missing a preferred URL:",
    f"{cordis_projects_full_df['preferred_project_url'].isna().sum():,}"
)

display(
    cordis_projects_full_df[
        [
            "project_id_clean",
            "project_website_url",
            "cordis_project_url",
            "preferred_project_url"
        ]
    ].head()
)

Projects with an external website: 267
Projects with an official CORDIS URL: 23,278
Projects missing a preferred URL: 0


,project_id_clean,project_website_url,cordis_project_url,preferred_project_url
0,101069359,<NA>,https://cordis.europa.eu/project/id/101069359,https://cordis.europa.eu/project/id/101069359
1,101069357,<NA>,https://cordis.europa.eu/project/id/101069357,https://cordis.europa.eu/project/id/101069357
2,101069586,<NA>,https://cordis.europa.eu/project/id/101069586,https://cordis.europa.eu/project/id/101069586
3,101069604,<NA>,https://cordis.europa.eu/project/id/101069604,https://cordis.europa.eu/project/id/101069604
4,101069529,<NA>,https://cordis.europa.eu/project/id/101069529,https://cordis.europa.eu/project/id/101069529


In [77]:
cordis_projects_full_df["search_text_core"] = (
    cordis_projects_full_df["search_text"]
)

search_columns = [
    "title",
    "objective",
    "keywords",
    "topics",
    "call_topic_title",
    "scivoc_titles"
]

cordis_projects_full_df["search_text"] = (
    cordis_projects_full_df[search_columns]
    .fillna("")
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print(
    "Search text created for:",
    f"{cordis_projects_full_df['search_text'].notna().sum():,} projects"
)

print(
    "Empty search-text rows:",
    cordis_projects_full_df["search_text"].eq("").sum()
)

print(
    "Projects with enriched EuroSciVoc text:",
    f"{cordis_projects_full_df['scivoc_titles'].notna().sum():,}"
)

print(
    "Median search-text length:",
    int(cordis_projects_full_df["search_text"].str.len().median())
)

Search text created for: 23,278 projects
Empty search-text rows: 0
Projects with enriched EuroSciVoc text: 20,062
Median search-text length: 2213


In [79]:
ai_pattern = (
    r"\b(?:"
    r"artificial intelligence|"
    r"machine learning|"
    r"deep learning|"
    r"neural networks?|"
    r"generative ai|"
    r"scientific machine learning|"
    r"reinforcement learning|"
    r"computer vision|"
    r"natural language processing|"
    r"foundation models?|"
    r"data[- ]driven|"
    r"predictive modelling|"
    r"predictive modeling|"
    r"ai"
    r")\b"
)

chemistry_materials_pattern = (
    r"\b(?:"
    r"chemistry|chemical|chemicals|"
    r"molecular|molecules?|"
    r"catalysis|catalysts?|catalytic|"
    r"polymers?|"
    r"materials science|materials discovery|advanced materials|"
    r"reaction prediction|chemical reactions?|"
    r"synthesis|synthetic chemistry|"
    r"computational chemistry|quantum chemistry|"
    r"electrochemistry|electrochemical|"
    r"spectroscopy|"
    r"drug discovery|"
    r"battery materials?|"
    r"nanomaterials?|"
    r"biomaterials?"
    r")\b"
)

cordis_projects_full_df["ai_keyword_match"] = (
    cordis_projects_full_df["search_text"]
    .str.contains(
        ai_pattern,
        case=False,
        na=False,
        regex=True
    )
)

cordis_projects_full_df["chemistry_materials_keyword_match"] = (
    cordis_projects_full_df["search_text"]
    .str.contains(
        chemistry_materials_pattern,
        case=False,
        na=False,
        regex=True
    )
)

both_keyword_match = (
    cordis_projects_full_df["ai_keyword_match"]
    & cordis_projects_full_df[
        "chemistry_materials_keyword_match"
    ]
)

cordis_projects_full_df["baseline_relevant"] = (
    cordis_projects_full_df["in_project_date_scope"]
    & both_keyword_match
)

print(
    "Projects matching AI terms:",
    f"{cordis_projects_full_df['ai_keyword_match'].sum():,}"
)

print(
    "Projects matching chemistry/materials terms:",
    f"{cordis_projects_full_df['chemistry_materials_keyword_match'].sum():,}"
)

print(
    "Projects matching both term groups:",
    f"{both_keyword_match.sum():,}"
)

print(
    "Baseline-relevant projects in 2021–2025:",
    f"{cordis_projects_full_df['baseline_relevant'].sum():,}"
)

Projects matching AI terms: 4,709
Projects matching chemistry/materials terms: 7,008
Projects matching both term groups: 1,009
Baseline-relevant projects in 2021–2025: 673


In [80]:
cordis_baseline_relevant_df = (
    cordis_projects_full_df.loc[
        cordis_projects_full_df["baseline_relevant"],
        [
            "project_id_clean",
            "record_year",
            "title",
            "objective",
            "call_topic_title",
            "scivoc_titles",
            "coordinator_name",
            "coordinator_country",
            "ecMaxContribution",
            "preferred_project_url"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Baseline candidate projects:",
    f"{len(cordis_baseline_relevant_df):,}"
)

display(
    cordis_baseline_relevant_df.sample(
        n=min(20, len(cordis_baseline_relevant_df)),
        random_state=42
    )[
        [
            "project_id_clean",
            "record_year",
            "title",
            "call_topic_title",
            "coordinator_country",
            "ecMaxContribution"
        ]
    ]
)

Baseline candidate projects: 673


,project_id_clean,record_year,title,call_topic_title,coordinator_country,ecMaxContribution
212,101113961,2023,AI-driven IP Intelligence Engine for Drug Disc...,Women TechEU,LV,75000.0
311,101130574,2024,Accelerated Discovery Nanobody Platform,EIC Pathfinder Open,ES,3315441.25
304,101130724,2024,QUANTUM-TOX - Revolutionizing Computational To...,EIC Pathfinder Open,IT,1994770.0
299,101135025,2024,A toolset for hyper-realistic and XR-based hum...,Next Generation eXtended Reality (RIA),ES,7655707.5
631,101178127,2025,ORGANIC BIOLOGICAL TRANSFORMATION OF ADDITIVE ...,Bio-intelligent manufacturing industries (Made...,ES,4994822.5
660,101219789,2025,A Helicopter View on Electrocatalysis,ERC STARTING GRANTS,DE,1486906.0
500,101163386,2025,INfrared FibeR Analysis of battery electroLYTe...,ERC STARTING GRANTS,FR,1500000.0
468,101141721,2024,Empowering Neural Rendering Methods with Physi...,ERC ADVANCED GRANTS,FR,2488029.0
109,101083748,2022,Highly Efficient Super Critical ZERO eMission ...,Next generation of renewable energy technologies,NL,2594660.0
602,101213674,2025,Revolutionising peptide-based drug discovery w...,ERC PROOF OF CONCEPT GRANTS,UK,150000.0


In [81]:
import re


def extract_pattern_matches(text, pattern):
    if pd.isna(text):
        return ""

    matches = {
        match.group(0).lower()
        for match in re.finditer(
            pattern,
            str(text),
            flags=re.IGNORECASE
        )
    }

    return " | ".join(sorted(matches))


relevance_review_sample_df = (
    cordis_projects_full_df.loc[
        cordis_projects_full_df["baseline_relevant"]
    ]
    .sample(
        n=20,
        random_state=42
    )
    .copy()
)

relevance_review_sample_df["ai_matches"] = (
    relevance_review_sample_df["search_text"]
    .apply(
        lambda text: extract_pattern_matches(
            text,
            ai_pattern
        )
    )
)

relevance_review_sample_df["chemistry_materials_matches"] = (
    relevance_review_sample_df["search_text"]
    .apply(
        lambda text: extract_pattern_matches(
            text,
            chemistry_materials_pattern
        )
    )
)

relevance_review_sample_df["objective_preview"] = (
    relevance_review_sample_df["objective"]
    .fillna("")
    .str.replace(r"\s+", " ", regex=True)
    .str.slice(0, 300)
)

display(
    relevance_review_sample_df[
        [
            "project_id_clean",
            "title",
            "ai_matches",
            "chemistry_materials_matches",
            "objective_preview"
        ]
    ]
)

,project_id_clean,title,ai_matches,chemistry_materials_matches,objective_preview
6916,101113961,AI-driven IP Intelligence Engine for Drug Disc...,ai | machine learning | natural language proce...,chemical | drug discovery | molecules,"Today, new drug development on average costs $..."
9775,101130574,Accelerated Discovery Nanobody Platform,ai | artificial intelligence,molecules,Since the approval of the first monoclonal ant...
9616,101130724,QUANTUM-TOX - Revolutionizing Computational To...,artificial intelligence,chemical | chemicals | molecular | quantum che...,Toxicology is at a crossroads. With ever more ...
9438,101135025,A toolset for hyper-realistic and XR-based hum...,ai,synthesis,The concept of presence can be understood as a...
17782,101178127,ORGANIC BIOLOGICAL TRANSFORMATION OF ADDITIVE ...,ai | artificial intelligence | data-driven,biomaterials,Biologicalisation considers the convergence of...
18683,101219789,A Helicopter View on Electrocatalysis,reinforcement learning,catalyst | catalysts | catalytic | chemical | ...,"Converting carbon dioxide, an unwanted by-prod..."
14170,101163386,INfrared FibeR Analysis of battery electroLYTe...,data-driven,chemical | chemistry | electrochemical | spect...,"""As global energy demands escalate, and the us..."
13466,101141721,Empowering Neural Rendering Methods with Physi...,deep learning,synthesis,While long restricted to an elite of expert di...
3931,101083748,Highly Efficient Super Critical ZERO eMission ...,machine learning,synthesis,Wind and sun will be central energy sources of...
16848,101213674,Revolutionising peptide-based drug discovery w...,ai,drug discovery | molecular | molecule | molecules,Peptide-based drugs offer distinct advantages ...


In [83]:
# Build classification text from the project's own description.
# EuroSciVoc and call-topic fields remain available for searching,
# but are excluded here because they can introduce broad incidental matches.

cordis_projects_full_df["relevance_text_core"] = (
    cordis_projects_full_df[
        ["title", "objective", "keywords"]
    ]
    .fillna("")
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)


# Strong AI terminology
strong_ai_pattern = (
    r"\b(?:"
    r"artificial intelligence|"
    r"machine learning|"
    r"deep learning|"
    r"neural networks?|"
    r"generative ai|"
    r"scientific machine learning|"
    r"reinforcement learning|"
    r"computer vision|"
    r"natural language processing|"
    r"foundation models?"
    r")\b"
)

full_ai_term_match = (
    cordis_projects_full_df["relevance_text_core"]
    .str.contains(
        strong_ai_pattern,
        case=False,
        na=False,
        regex=True
    )
)

# Match the uppercase acronym AI separately to reduce accidental matches.
ai_acronym_match = (
    cordis_projects_full_df["relevance_text_core"]
    .str.contains(
        r"\bAI(?:[- ](?:driven|enabled|based|assisted|powered))?\b",
        case=True,
        na=False,
        regex=True
    )
)

cordis_projects_full_df["strong_ai_match"] = (
    full_ai_term_match | ai_acronym_match
)


# More specific chemistry and materials terminology
strong_chemistry_materials_pattern = (
    r"\b(?:"
    r"chemistry|"
    r"chemical(?:s)?|"
    r"catalysis|catalysts?|catalytic|"
    r"polymers?|"
    r"materials science|"
    r"materials discovery|"
    r"advanced materials|"
    r"materials modell?ing|"
    r"materials simulation|"
    r"reaction prediction|"
    r"chemical reactions?|"
    r"chemical synthesis|"
    r"organic synthesis|"
    r"materials synthesis|"
    r"computational chemistry|"
    r"quantum chemistry|"
    r"molecular modell?ing|"
    r"molecular design|"
    r"molecular simulation|"
    r"molecular dynamics|"
    r"molecular property prediction|"
    r"electrochemistry|"
    r"electrochemical|"
    r"spectroscopy|"
    r"drug discovery|"
    r"battery materials?|"
    r"nanomaterials?|"
    r"biomaterials?"
    r")\b"
)

cordis_projects_full_df[
    "strong_chemistry_materials_match"
] = (
    cordis_projects_full_df["relevance_text_core"]
    .str.contains(
        strong_chemistry_materials_pattern,
        case=False,
        na=False,
        regex=True
    )
)


# Final refined relevance rule
cordis_projects_full_df["refined_relevant"] = (
    cordis_projects_full_df["in_project_date_scope"]
    & cordis_projects_full_df["strong_ai_match"]
    & cordis_projects_full_df[
        "strong_chemistry_materials_match"
    ]
)


# Calculate counts separately for Python 3.9 compatibility
original_candidate_count = (
    cordis_projects_full_df["baseline_relevant"].sum()
)

refined_candidate_count = (
    cordis_projects_full_df["refined_relevant"].sum()
)

removed_candidate_count = (
    cordis_projects_full_df["baseline_relevant"]
    & ~cordis_projects_full_df["refined_relevant"]
).sum()


print(
    "Original baseline candidates:",
    f"{original_candidate_count:,}"
)

print(
    "Refined candidates:",
    f"{refined_candidate_count:,}"
)

print(
    "Baseline candidates removed by refinement:",
    f"{removed_candidate_count:,}"
)


# Compare the same 20 reviewed projects against the refined rule
review_comparison_df = (
    cordis_projects_full_df.loc[
        relevance_review_sample_df.index,
        [
            "project_id_clean",
            "title",
            "strong_ai_match",
            "strong_chemistry_materials_match",
            "refined_relevant"
        ]
    ]
    .copy()
)

display(review_comparison_df)

Original baseline candidates: 673
Refined candidates: 416
Baseline candidates removed by refinement: 259


,project_id_clean,title,strong_ai_match,strong_chemistry_materials_match,refined_relevant
6916,101113961,AI-driven IP Intelligence Engine for Drug Disc...,True,True,True
9775,101130574,Accelerated Discovery Nanobody Platform,True,False,False
9616,101130724,QUANTUM-TOX - Revolutionizing Computational To...,True,True,True
9438,101135025,A toolset for hyper-realistic and XR-based hum...,True,False,False
17782,101178127,ORGANIC BIOLOGICAL TRANSFORMATION OF ADDITIVE ...,True,True,True
18683,101219789,A Helicopter View on Electrocatalysis,True,True,True
14170,101163386,INfrared FibeR Analysis of battery electroLYTe...,False,True,False
13466,101141721,Empowering Neural Rendering Methods with Physi...,True,False,False
3931,101083748,Highly Efficient Super Critical ZERO eMission ...,True,False,False
16848,101213674,Revolutionising peptide-based drug discovery w...,True,True,True


In [84]:
cordis_projects_full_df["relevance_tier"] = "out_of_scope"

cordis_projects_full_df.loc[
    cordis_projects_full_df["baseline_relevant"],
    "relevance_tier"
] = "broad_match"

cordis_projects_full_df.loc[
    cordis_projects_full_df["refined_relevant"],
    "relevance_tier"
] = "core_match"


relevance_tier_counts = (
    cordis_projects_full_df.loc[
        cordis_projects_full_df["in_project_date_scope"],
        "relevance_tier"
    ]
    .value_counts()
    .rename_axis("relevance_tier")
    .reset_index(name="project_count")
)

refined_only_count = (
    cordis_projects_full_df["refined_relevant"]
    & ~cordis_projects_full_df["baseline_relevant"]
).sum()

baseline_only_count = (
    cordis_projects_full_df["baseline_relevant"]
    & ~cordis_projects_full_df["refined_relevant"]
).sum()

both_filter_count = (
    cordis_projects_full_df["baseline_relevant"]
    & cordis_projects_full_df["refined_relevant"]
).sum()

display(relevance_tier_counts)

print(
    "Projects matching both filters:",
    f"{both_filter_count:,}"
)

print(
    "Broad-filter-only projects:",
    f"{baseline_only_count:,}"
)

print(
    "Refined-filter-only projects:",
    f"{refined_only_count:,}"
)

,relevance_tier,project_count
0,out_of_scope,17256
1,core_match,416
2,broad_match,259


Projects matching both filters: 414
Broad-filter-only projects: 259
Refined-filter-only projects: 2


In [85]:
refined_only_projects_df = (
    cordis_projects_full_df.loc[
        cordis_projects_full_df["refined_relevant"]
        & ~cordis_projects_full_df["baseline_relevant"],
        [
            "project_id_clean",
            "record_year",
            "title",
            "objective",
            "keywords",
            "call_topic_title",
            "strong_ai_match",
            "strong_chemistry_materials_match"
        ]
    ]
    .copy()
)

refined_only_projects_df["objective_preview"] = (
    refined_only_projects_df["objective"]
    .fillna("")
    .str.replace(r"\s+", " ", regex=True)
    .str.slice(0, 400)
)

display(
    refined_only_projects_df[
        [
            "project_id_clean",
            "record_year",
            "title",
            "call_topic_title",
            "keywords",
            "objective_preview"
        ]
    ]
)

,project_id_clean,record_year,title,call_topic_title,keywords,objective_preview
1225,101065605,2022,Voltage-Controlled Electronic and Magnetic Pha...,MSCA Postdoctoral Fellowships 2021,"Mott insulators, Non-Equilibrium Green’s funct...",Matter can exhibit a complicated phase diagram...
15287,101167207,2025,Mechanical characterization of soft tissue in ...,ERC SYNERGY GRANTS,"Biomechanics, Solid Mechanics, Materials Model...",Computational biomechanics is a fast-growing a...


In [87]:
refined_only_term_review_df = refined_only_projects_df.copy()

# Rebuild the same relevance text inside this smaller review table
refined_only_term_review_df["relevance_text_core"] = (
    refined_only_term_review_df[
        ["title", "objective", "keywords"]
    ]
    .fillna("")
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)


def extract_case_sensitive_matches(text, pattern):
    if pd.isna(text):
        return ""

    matches = {
        match.group(0)
        for match in re.finditer(
            pattern,
            str(text)
        )
    }

    return " | ".join(sorted(matches))


refined_only_term_review_df["ai_matches"] = (
    refined_only_term_review_df["relevance_text_core"]
    .apply(
        lambda text: extract_pattern_matches(
            text,
            strong_ai_pattern
        )
    )
)

refined_only_term_review_df["ai_acronym_matches"] = (
    refined_only_term_review_df["relevance_text_core"]
    .apply(
        lambda text: extract_case_sensitive_matches(
            text,
            r"\bAI(?:[- ](?:driven|enabled|based|assisted|powered))?\b"
        )
    )
)

refined_only_term_review_df["chemistry_materials_matches"] = (
    refined_only_term_review_df["relevance_text_core"]
    .apply(
        lambda text: extract_pattern_matches(
            text,
            strong_chemistry_materials_pattern
        )
    )
)

display(
    refined_only_term_review_df[
        [
            "project_id_clean",
            "title",
            "ai_matches",
            "ai_acronym_matches",
            "chemistry_materials_matches",
            "objective_preview"
        ]
    ]
)

,project_id_clean,title,ai_matches,ai_acronym_matches,chemistry_materials_matches,objective_preview
1225,101065605,Voltage-Controlled Electronic and Magnetic Pha...,machine learning,,materials modeling,Matter can exhibit a complicated phase diagram...
15287,101167207,Mechanical characterization of soft tissue in ...,machine learning | neural networks,,materials modeling,Computational biomechanics is a fast-growing a...


In [89]:
relevance_tier_order = pd.CategoricalDtype(
    categories=[
        "core_match",
        "broad_match",
        "out_of_scope"
    ],
    ordered=True
)

cordis_projects_full_df["relevance_tier"] = (
    cordis_projects_full_df["relevance_tier"]
    .astype(relevance_tier_order)
)

cordis_candidate_projects_df["relevance_tier"] = (
    cordis_candidate_projects_df["relevance_tier"]
    .astype(relevance_tier_order)
)

cordis_candidate_projects_df = (
    cordis_candidate_projects_df
    .sort_values(
        [
            "relevance_tier",
            "record_year",
            "ecMaxContribution"
        ],
        ascending=[True, False, False]
    )
    .reset_index(drop=True)
)

candidate_year_tier_summary = (
    cordis_candidate_projects_df
    .groupby(
        ["record_year", "relevance_tier"],
        observed=True
    )
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

display(candidate_year_tier_summary)

display(
    cordis_candidate_projects_df[
        [
            "project_id_clean",
            "record_year",
            "title",
            "relevance_tier",
            "coordinator_country",
            "ecMaxContribution"
        ]
    ].head(10)
)

relevance_tier,record_year,core_match,broad_match
0,2022,66,43
1,2023,95,69
2,2024,135,77
3,2025,120,70


,project_id_clean,record_year,title,relevance_tier,coordinator_country,ecMaxContribution
0,101252959,2025,Protein-ligand data generation at scale to sup...,core_match,UK,29897910.0
1,101192848,2025,"FULLy integrated, autonomous & chemistry agnos...",core_match,BE,19949437.4
2,101136570,2025,Translational Research and Innovation in Ophth...,core_match,PL,15000000.0
3,101234224,2025,Italy for Artificial Intelligence,core_match,IT,15000000.0
4,101260391,2025,AI Factory France,core_match,FR,14978231.0
5,101167540,2025,Active Hybrid Photonic Integrated Circuits for...,core_match,DE,13999999.0
6,101167416,2025,Deep learning analysis of imaging and metabolo...,core_match,FR,10968734.0
7,101167045,2025,Concrete matrices for high-cycle-fatigue resis...,core_match,CZ,9993698.0
8,101167207,2025,Mechanical characterization of soft tissue in ...,core_match,DE,9991449.0
9,101234349,2025,The Swedish AI Innovation Factory,core_match,SE,9792375.0


In [90]:
projects_2021_check_df = (
    cordis_projects_full_df.loc[
        cordis_projects_full_df["record_year"].eq(2021),
        [
            "project_id_clean",
            "title",
            "ecSignatureDate",
            "startDate",
            "ai_keyword_match",
            "chemistry_materials_keyword_match",
            "strong_ai_match",
            "strong_chemistry_materials_match",
            "relevance_tier"
        ]
    ]
    .copy()
)

print("All projects starting in 2021:", len(projects_2021_check_df))

print(
    "2021 projects matching broad AI terms:",
    projects_2021_check_df["ai_keyword_match"].sum()
)

print(
    "2021 projects matching broad chemistry/materials terms:",
    projects_2021_check_df[
        "chemistry_materials_keyword_match"
    ].sum()
)

print(
    "2021 projects matching both broad term groups:",
    (
        projects_2021_check_df["ai_keyword_match"]
        & projects_2021_check_df[
            "chemistry_materials_keyword_match"
        ]
    ).sum()
)

display(
    projects_2021_check_df[
        [
            "project_id_clean",
            "title",
            "ecSignatureDate",
            "startDate",
            "relevance_tier"
        ]
    ]
)

All projects starting in 2021: 30
2021 projects matching broad AI terms: 3
2021 projects matching broad chemistry/materials terms: 0
2021 projects matching both broad term groups: 0


,project_id_clean,title,ecSignatureDate,startDate,relevance_tier
43,101039221,Support to the Vice-Presidents of the ERC Scie...,2021-07-07,2021-04-01,out_of_scope
92,101046041,EU-Africa Concerted Action on SAR-CoV-2 Virus ...,2021-11-12,2021-11-01,out_of_scope
93,101045989,SARS-coV2 variants Evaluation in pRegnancy and...,2021-11-11,2021-11-01,out_of_scope
94,101046016,European Cohorts of Patients and Schools to Ad...,2021-11-15,2021-10-14,out_of_scope
497,101045949,"ExeVir's XVR011, a best in class nanobody-base...",2021-11-15,2021-12-01,out_of_scope
532,101052304,Development of indicators & econometric analys...,2021-10-27,2021-11-01,out_of_scope
535,101052416,COST: Europe's most empowering research progra...,2021-11-25,2021-11-01,out_of_scope
542,101046109,European Clinical Research Alliance on Infecti...,2021-11-12,2021-12-01,out_of_scope
554,101052293,MSCA fostering balanced mobility flows in Europe,2021-09-23,2021-10-01,out_of_scope
1628,190126069,A biorefinery for upcycling coffee waste into ...,2022-05-05,2021-11-01,out_of_scope


In [91]:
from pathlib import Path

cordis_processed_dir = (
    Path(PROJECT_ROOT)
    / "Data"
    / "Processed_Data"
    / "CORDIS"
)

cordis_processed_dir.mkdir(
    parents=True,
    exist_ok=True
)

full_output_path = (
    cordis_processed_dir
    / "cordis_projects_enriched_2021_2025.csv"
)

candidate_output_path = (
    cordis_processed_dir
    / "cordis_candidate_projects_2021_2025.csv"
)

cordis_projects_full_df.to_csv(
    full_output_path,
    index=False,
    encoding="utf-8-sig"
)

cordis_candidate_projects_df.to_csv(
    candidate_output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Full dataset saved:")
print(full_output_path)
print(
    "Rows × columns:",
    cordis_projects_full_df.shape
)
print(
    "File size:",
    f"{full_output_path.stat().st_size / 1_000_000:.2f} MB"
)

print("\nCandidate dataset saved:")
print(candidate_output_path)
print(
    "Rows × columns:",
    cordis_candidate_projects_df.shape
)
print(
    "File size:",
    f"{candidate_output_path.stat().st_size / 1_000_000:.2f} MB"
)

Full dataset saved:
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\GrantScopeAI\Data\Processed_Data\CORDIS\cordis_projects_enriched_2021_2025.csv
Rows × columns: (23278, 75)
File size: 217.97 MB

Candidate dataset saved:
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\GrantScopeAI\Data\Processed_Data\CORDIS\cordis_candidate_projects_2021_2025.csv
Rows × columns: (675, 75)
File size: 6.52 MB
